# Accurate but Overconfident? — RUL accuracy vs. interval calibration (C-MAPSS)

This notebook reproduces the main study of this project from scratch:

1. installs the libraries,
2. writes the exact project code into `/content/RUL_project/`,
3. downloads the NASA C-MAPSS files (FD001, FD004) and checks them against the recorded checksums,
4. trains Ridge, gradient boosting and an LSTM for 2 subsets × 5 seeds (about 10–20 minutes on the free CPU runtime),
5. computes all metrics, hypothesis tests and figures,
6. runs the exploratory post-hoc recalibration analysis (deviation D7),
7. runs the exploratory deep-ensemble, MC-dropout and adaptive-conformal analysis (deviation D8), and offers the results as a zip download.

**Run it with:** `Runtime → Run all`. A GPU is not needed; the code runs on the CPU.

Small differences from the reported numbers (in the last decimals) are possible if Colab's library versions differ from the ones used for the reported results (PyTorch 2.15, LightGBM 4.7, scikit-learn 1.9).

## 1. Setup

In [ ]:
!pip -q install lightgbm huggingface_hub
import os, sys, platform
import numpy as np, pandas as pd, torch, sklearn, lightgbm, scipy, matplotlib
print("Python", platform.python_version(), "| torch", torch.__version__, "| sklearn", sklearn.__version__,
      "| lightgbm", lightgbm.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
os.makedirs("/content/RUL_project/src", exist_ok=True)
os.makedirs("/content/RUL_project/main", exist_ok=True)
os.makedirs("/content/RUL_project/data_raw", exist_ok=True)

## 2. Project code
Shared modules (`src/`): data loading and windows, model utilities, engine-weighted conformal prediction.

In [ ]:
%%writefile /content/RUL_project/src/data.py
"""C-MAPSS data loading, regime-wise normalisation and windowing.

Design rules (enforced here, audited in review):
  * All splits are by engine unit, never by window.
  * Normalisation statistics are fit on training units only (per fold).
  * Regimes unseen in the source are normalised with statistics from the
    *unlabelled* target inputs (label-free, transductive). This is stated
    explicitly in the write-up as a form of unsupervised alignment.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path(__file__).resolve().parents[1] / "data_raw"
COLS = ["unit", "cycle", "s1", "s2", "s3"] + [f"x{i}" for i in range(1, 22)]
# 14 informative sensors used throughout the C-MAPSS literature
SENSORS = [f"x{i}" for i in (2, 3, 4, 7, 8, 9, 11, 12, 13, 14, 15, 17, 20, 21)]
ALTITUDES = np.array([0.0, 10.0, 20.0, 25.0, 35.0, 42.0])  # kft, one per regime
RUL_CAP = 125
WINDOW = 30


def load(fd: str, split: str) -> pd.DataFrame:
    df = pd.read_csv(RAW / f"{split}_{fd}.txt", sep=r"\s+", header=None, names=COLS)
    df["regime"] = np.abs(df["s1"].values[:, None] - ALTITUDES[None, :]).argmin(1)
    if split == "train":
        eol = df.groupby("unit")["cycle"].transform("max")
        df["rul"] = eol - df["cycle"]
    else:
        rul = pd.read_csv(RAW / f"RUL_{fd}.txt", header=None)[0].values
        eol = df.groupby("unit")["cycle"].transform("max")
        df["rul"] = (eol - df["cycle"]) + rul[df["unit"].values - 1]
    df["rul_capped"] = df["rul"].clip(upper=RUL_CAP)
    return df


def regime_stats(df: pd.DataFrame) -> dict:
    g = df.groupby("regime")[SENSORS]
    mu, sd = g.mean(), g.std().replace(0, 1.0).fillna(1.0)
    return {r: (mu.loc[r].values, sd.loc[r].values) for r in mu.index}


def normalise(df: pd.DataFrame, src_stats: dict, tgt_stats: dict | None = None):
    """Per-regime z-score. Source stats preferred; unseen regimes fall back to
    label-free target stats. Returns array and fraction of rows using fallback."""
    X = df[SENSORS].values.astype(np.float32).copy()
    fallback = 0
    for r in np.unique(df["regime"].values):
        m = df["regime"].values == r
        if r in src_stats:
            mu, sd = src_stats[r]
        elif tgt_stats is not None and r in tgt_stats:
            mu, sd = tgt_stats[r]
            fallback += m.sum()
        else:
            raise ValueError(f"No statistics for regime {r}")
        X[m] = (X[m] - mu) / sd
    return X, fallback / len(df)


def windows(df: pd.DataFrame, X: np.ndarray, last_only: bool = False):
    """Sliding windows per unit; units shorter than WINDOW are edge-padded with
    their first observation. Returns (Xw, y_capped, y_raw, unit_ids, rul_true)."""
    xs, yc, yr, us = [], [], [], []
    for u, idx in df.groupby("unit").indices.items():
        xu = X[idx]
        if len(xu) < WINDOW:
            xu = np.vstack([np.repeat(xu[:1], WINDOW - len(xu), 0), xu])
            pad = WINDOW - len(idx)
        else:
            pad = 0
        rc = df["rul_capped"].values[idx]
        rr = df["rul"].values[idx]
        ends = [len(xu)] if last_only else range(WINDOW, len(xu) + 1)
        for e in ends:
            xs.append(xu[e - WINDOW:e])
            j = e - 1 - pad  # index into the original (unpadded) rows
            yc.append(rc[j]); yr.append(rr[j]); us.append(u)
    return (np.stack(xs).astype(np.float32), np.array(yc, np.float32),
            np.array(yr, np.float32), np.array(us))


def unit_folds(units: np.ndarray, k: int, seed: int) -> list[np.ndarray]:
    rng = np.random.default_rng(seed)
    u = rng.permutation(np.unique(units))
    return np.array_split(u, k)

In [ ]:
%%writefile /content/RUL_project/src/models.py
"""Models: 1D-CNN with heteroscedastic Gaussian head, and LightGBM quantile baseline.

v2 changes (after review): dropout in every conv block (needed for MC dropout to
perturb the whole feature extractor); early stopping on validation Gaussian NLL
(a proper scoring rule) instead of RMSE, so the variance head is selected for;
architecture width/dropout/beta chosen by a documented grid on source-validation
units only (tune.py); LightGBM uses early stopping on validation pinball loss.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn

from data import RUL_CAP

SCALE = float(RUL_CAP)  # targets are trained on y / 125
CFG = {"width": 32, "p": 0.1, "beta": 0.5, "lr": 1e-3}  # selected by tune.py (results/tuning.json)


class CNN(nn.Module):
    """Small temporal CNN in the spirit of Li et al. (2018), with a mean and a
    log-variance head."""

    def __init__(self, n_feat: int, width: int = 64, p: float = 0.1):
        super().__init__()
        w = width
        self.body = nn.Sequential(
            nn.Conv1d(n_feat, w, 5, padding=2), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(w, w, 5, padding=2), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(w, w, 3, padding=1), nn.ReLU(), nn.Dropout(p),
            nn.AdaptiveAvgPool1d(4), nn.Flatten(),
            nn.Linear(4 * w, 64), nn.ReLU(), nn.Dropout(p),
        )
        self.mu = nn.Linear(64, 1)
        self.logvar = nn.Linear(64, 1)

    def forward(self, x):  # x: (B, T, F)
        h = self.body(x.transpose(1, 2))
        return self.mu(h).squeeze(-1), self.logvar(h).squeeze(-1).clamp(-10, 6)


def beta_nll(mu, logvar, y, beta=0.5):
    """beta-NLL (Seitzer et al., ICLR 2022): plain Gaussian NLL lets the variance
    absorb hard points and starves the mean of gradient; beta in (0,1] rebalances.
    beta=0 recovers the standard NLL."""
    var = logvar.exp()
    nll = 0.5 * (logvar + (y - mu) ** 2 / var)
    return (nll * var.detach() ** beta).mean() if beta > 0 else nll.mean()


def _val_nll(m, Xva_t, yva):
    with torch.no_grad():
        mu, lv = m(Xva_t)
    mu, var = mu.numpy() * SCALE, lv.exp().numpy() * SCALE ** 2
    nll = float(np.mean(0.5 * np.log(2 * np.pi * var) + 0.5 * (yva - mu) ** 2 / var))
    rmse = float(np.sqrt(np.mean((mu - yva) ** 2)))
    return nll, rmse


def train_cnn(Xtr, ytr, Xva, yva, seed, cfg=None, max_epochs=60, patience=8, bs=256):
    cfg = {**CFG, **(cfg or {})}
    torch.manual_seed(seed); np.random.seed(seed)
    m = CNN(Xtr.shape[-1], cfg["width"], cfg["p"])
    opt = torch.optim.Adam(m.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    Xtr_t, ytr_t = torch.from_numpy(Xtr), torch.from_numpy(ytr / SCALE)
    Xva_t = torch.from_numpy(Xva)
    best, best_state, bad, ep = np.inf, None, 0, 0
    g = torch.Generator().manual_seed(seed)
    for ep in range(max_epochs):
        m.train()
        perm = torch.randperm(len(Xtr_t), generator=g)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            mu, lv = m(Xtr_t[b])
            loss = beta_nll(mu, lv, ytr_t[b], cfg["beta"])
            opt.zero_grad(); loss.backward(); opt.step()
        m.eval()
        nll, rmse = _val_nll(m, Xva_t, yva)
        if nll < best - 1e-4:
            best, bad, best_rmse = nll, 0, rmse
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    m.load_state_dict(best_state)
    m.eval()
    return m, {"val_nll": best, "val_rmse": best_rmse, "epochs": ep + 1}


@torch.no_grad()
def predict(m, X, bs=4096):
    m.eval()
    out_mu, out_var = [], []
    for i in range(0, len(X), bs):
        mu, lv = m(torch.from_numpy(X[i:i + bs]))
        out_mu.append(mu.numpy() * SCALE); out_var.append(lv.exp().numpy() * SCALE ** 2)
    return np.concatenate(out_mu), np.concatenate(out_var)


@torch.no_grad()
def predict_mc(m, X, T=100, seed=0, bs=4096):
    """MC dropout (Gal & Ghahramani, 2016) on a heteroscedastic net: dropout active,
    T passes. Returns per-pass means (T, n) and aleatoric variances (T, n) so that
    T-sensitivity can be computed afterwards without re-running."""
    torch.manual_seed(seed)
    m.train()  # dropout on (no batch-norm in the model)
    mus, vs = [], []
    for _ in range(T):
        mu_b, v_b = [], []
        for i in range(0, len(X), bs):
            mu, lv = m(torch.from_numpy(X[i:i + bs]))
            mu_b.append(mu.numpy() * SCALE); v_b.append(lv.exp().numpy() * SCALE ** 2)
        mus.append(np.concatenate(mu_b)); vs.append(np.concatenate(v_b))
    m.eval()
    return np.stack(mus).astype(np.float32), np.stack(vs).astype(np.float32)


# ---------------- LightGBM ----------------
def handcrafted(Xw):
    """Window features: last value, mean, std, OLS slope per sensor."""
    t = np.arange(Xw.shape[1], dtype=np.float32); t = (t - t.mean()) / ((t - t.mean()) ** 2).sum()
    slope = np.einsum("t,ntf->nf", t, Xw)
    return np.concatenate([Xw[:, -1], Xw.mean(1), Xw.std(1), slope], 1)


def train_lgbm_quantiles(F, y, Fva, yva, seed, qs=(0.05, 0.5, 0.95)):
    """Quantile GBMs with early stopping on validation pinball loss (same
    validation units as the CNN)."""
    import lightgbm as lgb
    models = {}
    for q in qs:
        models[q] = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=2000,
                                      learning_rate=0.03, num_leaves=31, min_child_samples=50,
                                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                      random_state=seed, verbose=-1, n_jobs=2).fit(
            F, y, eval_set=[(Fva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
    return models


def domain_logit(F_src, F_tgt, F_query_list, seed):
    """Probabilistic domain classifier source(0) vs target(1) on unlabelled inputs,
    for weighted conformal prediction (Tibshirani et al., 2019). Returns the
    log-odds for each query array, rebalanced for class sizes."""
    import lightgbm as lgb
    X = np.concatenate([F_src, F_tgt]); y = np.r_[np.zeros(len(F_src)), np.ones(len(F_tgt))]
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                             subsample=0.8, subsample_freq=1, random_state=seed, verbose=-1, n_jobs=2).fit(X, y)
    prior = np.log(len(F_src) / len(F_tgt))
    return [clf.predict(Fq, raw_score=True) + prior for Fq in F_query_list]

In [ ]:
%%writefile /content/RUL_project/src/conformal.py
"""Interval constructions (v2, revised after independent review).

Exchangeable objects are ENGINES, not windows. Every conformal variant below is
written in one 'augmented weighted distribution' form:

    Q_{1-a}( sum_i p_i * delta_{V_i} + p_test * delta_{+inf} )

where calibration point i carries weight w_i (unit-balanced: each engine has
total weight 1, i.e. the CDF-pooling estimator of Dunn, Wasserman & Ramdas,
2022) and the test engine carries weight w_test (1 when unweighted, or the
likelihood ratio for weighted CP, Tibshirani et al. 2019). The +inf atom makes
the finite-sample correction exact, including the regime where the valid
interval is infinite (e.g. k=5 calibration engines at a=0.1) -- this fixes
review issue C2.

Guarantee caveats (documented in the write-up):
  * CV+ (Barber et al., 2021) guarantees 1-2a in the worst case (~1-a in practice).
  * Unit-balanced weighting targets a test point drawn uniformly over the windows of
    a fresh engine. NASA's official test truncation is a different distribution
    (review C3), so on the official test set coverage is NOT guaranteed even in
    domain; protocol B evaluates under the matched distribution.
"""
from __future__ import annotations

import numpy as np
from scipy.stats import norm

CHUNK = 256


def unit_weights(units: np.ndarray) -> np.ndarray:
    _, inv, cnt = np.unique(units, return_inverse=True, return_counts=True)
    return 1.0 / cnt[inv]


def _wq_aug(V, W, w_test, q):
    """Upper weighted quantile at level q of rows of V (n_t, n_c) with weights
    W ((n_c,) or (n_t, n_c)) plus an atom at +inf with weight w_test (n_t,)."""
    n_t = V.shape[0]
    W = np.broadcast_to(W, V.shape)
    out = np.empty(n_t)
    for s in range(0, n_t, CHUNK):
        v, w, wt = V[s:s + CHUNK], W[s:s + CHUNK], w_test[s:s + CHUNK]
        o = np.argsort(v, -1)
        vs, cw = np.take_along_axis(v, o, -1), np.cumsum(np.take_along_axis(w, o, -1), -1)
        tot = cw[:, -1] + wt
        cw = cw / tot[:, None]
        idx = (cw < q - 1e-12).sum(-1)
        res = np.full(len(v), np.inf)                      # quantile falls on the +inf atom
        ok = idx < v.shape[1]
        res[ok] = vs[ok, idx[ok]]
        out[s:s + CHUNK] = res
    return out


def cv_interval(L, U, W, w_test, alpha):
    """Generic CV+/split interval: lower = Q_a{L_i, -inf}, upper = Q_{1-a}{U_i, +inf}."""
    hi = _wq_aug(U, W, w_test, 1 - alpha)
    lo = -_wq_aug(-L, W, w_test, 1 - alpha)
    return lo, hi


# ---------------- Gaussian (model-based) intervals ----------------
def gauss_interval(mu, var, alpha):
    z = norm.ppf(1 - alpha / 2)
    s = np.sqrt(var)
    return mu - z * s, mu + z * s


def mixture_moments(mu_k, var_k):
    """Moments of an equally weighted Gaussian mixture over members (rows)."""
    return mu_k.mean(0), var_k.mean(0) + mu_k.var(0)


# ---------------- CV+ variants (fold-model predictions mu_k: (K, n_test)) ----------------
def cvplus_abs(mu_k, oof_mu, oof_y, fold, w, alpha, w_test=None, W=None):
    R = np.abs(oof_y - oof_mu)
    w_test = np.ones(mu_k.shape[1]) if w_test is None else w_test
    W = w if W is None else W
    return cv_interval(mu_k[fold].T - R, mu_k[fold].T + R, W, w_test, alpha)


def cvplus_norm(mu_k, sd_k, oof_mu, oof_sd, oof_y, fold, w, alpha):
    S = np.abs(oof_y - oof_mu) / oof_sd
    L = (mu_k[fold] - S[:, None] * sd_k[fold]).T
    U = (mu_k[fold] + S[:, None] * sd_k[fold]).T
    return cv_interval(L, U, w, np.ones(mu_k.shape[1]), alpha)


def cvplus_cqr(qlo_k, qhi_k, oof_qlo, oof_qhi, oof_y, fold, w, alpha):
    E = np.maximum(oof_qlo - oof_y, oof_y - oof_qhi)
    return cv_interval(qlo_k[fold].T - E, qhi_k[fold].T + E, w, np.ones(qlo_k.shape[1]), alpha)


def mondrian_cvplus_abs(mu_k, oof_mu, oof_y, fold, w, alpha, bins):
    """Horizon-conditional (Mondrian) CV+, fixed per review C4: the category of the
    pair (test x, calibration i) is the predicted-RUL bin of the SAME fold model
    k(i) on both points; weights are the global unit weights restricted to the
    category. Valid conditionally on the predicted-RUL bin; report coverage by
    PREDICTED bin, not true-RUL bin."""
    b_cal = np.digitize(oof_mu, bins)                 # bin of mu_{k(i)}(x_i)
    b_test = np.digitize(mu_k[fold].T, bins)           # (n_t, n_c): bin of mu_{k(i)}(x)
    Wm = np.where(b_test == b_cal[None, :], w[None, :], 0.0)
    return cvplus_abs(mu_k, oof_mu, oof_y, fold, w, alpha, W=Wm)


def weighted_cvplus_abs(mu_k, oof_mu, oof_y, fold, w, logit_cal, logit_test, alpha, clip=8.0):
    """Weighted CP under covariate shift (Tibshirani et al., 2019) in CV+ form.
    Likelihood ratio r(x) = exp(domain-classifier log-odds), clipped to [e^-8, e^8]
    for numerical stability. Calibration weight = unit weight x r(x_i); test
    weight = r(x) (one engine). Returns interval and the effective sample size of
    the calibration weights (unit-level Kish ESS)."""
    r_cal = np.exp(np.clip(logit_cal, -clip, clip))
    r_test = np.exp(np.clip(logit_test, -clip, clip))
    Wc = w * r_cal
    lo, hi = cvplus_abs(mu_k, oof_mu, oof_y, fold, w, alpha, w_test=r_test, W=Wc)
    ess = Wc.sum() ** 2 / (Wc ** 2).sum()
    return lo, hi, ess


# ---------------- split conformal (k-shot target recalibration) ----------------
def split_abs(mu_test, mu_cal, y_cal, units_cal, alpha):
    R = np.abs(y_cal - mu_cal)
    n = len(mu_test)
    return cv_interval(mu_test[:, None] - R[None, :], mu_test[:, None] + R[None, :],
                       unit_weights(units_cal), np.ones(n), alpha)


def split_norm(mu_test, sd_test, mu_cal, sd_cal, y_cal, units_cal, alpha):
    S = np.abs(y_cal - mu_cal) / sd_cal
    n = len(mu_test)
    return cv_interval(mu_test[:, None] - S[None, :] * sd_test[:, None],
                       mu_test[:, None] + S[None, :] * sd_test[:, None],
                       unit_weights(units_cal), np.ones(n), alpha)

Main study (`main/`): the pre-registration, the experiment, the analysis, the hypothesis tests and the supporting checks.

In [ ]:
# Exact copies of the pre-registration and the deviations log (written byte-for-byte so the checksum can be verified)
open('/content/RUL_project/main/PREREGISTRATION.md', 'w', encoding='utf-8').write("# Pre-registration — main study\n\n**Written:** 2026-11-16, before any Ridge or LSTM model was trained and before any main-study result existed.\n**Frozen by:** SHA-256 of this file, recorded in `PREREGISTRATION.sha256` at the time of writing.\nAny change after that point is listed under *Deviations* at the bottom, with date and reason.\n\n## Research question\n\nDo RUL models with lower point-prediction error also give better prediction intervals?\nTwo parts:\n\n- **Native intervals:** the uncertainty each model produces by itself.\n- **Conformal intervals:** the same model wrapped in one common method, split conformal prediction.\n\n## Status of the data (honest disclosure)\n\nA pilot study (10 Nov 2026) already evaluated a 1D-CNN and LightGBM (quantile regression and CQR) on the official test sets of all four C-MAPSS subsets, including FD001 and FD004. Therefore:\n\n- **FD001** results are reported as **exploratory** (replication of the pilot setting).\n- **FD004** results are **pre-specified but not blind**: the settings below are fixed before running, but LightGBM-type results on this test set were seen in the pilot.\n- **Ridge and LSTM** have never been run on any subset. For these two models both subsets are unseen.\n\n## Data\n\n- Subsets: FD001 (1 operating condition, 1 fault mode) and FD004 (6 conditions, 2 fault modes).\n- 14 sensors: x2, x3, x4, x7, x8, x9, x11, x12, x13, x14, x15, x17, x20, x21.\n- Regime from altitude setting; z-score per regime using statistics of the training engines of each seed.\n- Target: RUL capped at 125 cycles (piecewise linear). Window: 30 cycles. Test engines shorter than 30 cycles are edge-padded.\n\n## Splits (per seed s ∈ {0, 1, 2, 3, 4})\n\n- Engines of the official training file are shuffled with seed s.\n- **70% training engines**; of these, 15% are held out as **validation engines** for early stopping and Ridge α.\n- **30% calibration engines**, used only for split conformal.\n- **Test set:** the official NASA test file, one prediction per engine at its last cycle (FD001: 100 engines, FD004: 248).\n- No engine appears in more than one of training, validation, calibration.\n\n## Models\n\n| Model | Input | Point prediction | Native interval |\n|---|---|---|---|\n| Ridge | 56 window features (last value, mean, s.d., OLS slope per sensor), standardised | Ridge regression, α ∈ {0.01, 0.1, 1, 10, 100, 1000} chosen by validation MSE | Gaussian, homoscedastic: σ = RMSE on validation engines |\n| Gradient boosting (LightGBM) | Same 56 features | L2 objective, learning rate 0.03, 31 leaves, min 50 samples per leaf, subsample 0.8, colsample 0.8, ≤ 2000 trees, early stopping (50 rounds) on validation loss | Quantile regression with identical settings at q ∈ {0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.75, 0.80, 0.85, 0.90, 0.95, 0.975}; crossing quantiles sorted |\n| Small LSTM | Raw 30 × 14 window | 2-layer LSTM, 32 hidden units, dropout 0.1 between layers, dense 32 → mean and log-variance; β-NLL (β = 0.5); Adam lr 1e-3, weight decay 1e-4, batch 256, ≤ 60 epochs, early stopping on validation Gaussian NLL (patience 8) | Gaussian, heteroscedastic (predicted variance) |\n\nNo hyperparameter is tuned on calibration or test engines. No setting will be changed after test results are seen.\n\n## Interval methods\n\n- **Native** as in the table above.\n- **Split conformal** (identical for all models): score = |y − ŷ| on all windows of the calibration engines, each engine given total weight 1; interval ŷ ± q, where q is the engine-weighted quantile at level (1 − α)(1 + 1/n_engines), returning an infinite interval when that level exceeds 1.\n- Points and intervals are intersected with the target support [0, 125] for evaluation; unclipped values are also stored.\n\n## Metrics\n\n- Accuracy: **MAE**, **RMSE** (primary); mean NASA score on raw predictions vs official RUL (secondary).\n- Calibration at nominal levels L ∈ {0.50, 0.60, 0.70, 0.80, 0.90, 0.95}, highlighted 0.80 / 0.90 / 0.95:\n  - **PICP** (empirical coverage), **MPIW** (mean width), **interval score** (Winkler).\n  - **Calibration curve**: PICP against L.\n  - **Calibration error (CE)**: mean |PICP − L| over the six levels.\n- Exploratory: coverage by true RUL band (≤ 40, 40–80, 80–125); coverage on test engines with true RUL < 125.\n\n## Statistics\n\n- Mean and seed s.d. over 5 seeds.\n- 95% intervals by engine-level bootstrap (2,000 resamples) of per-engine values averaged over seeds.\n- Model comparisons by paired engine-level bootstrap; results described per subset; no p-value thresholds beyond the 95% intervals.\n\n## Hypotheses\n\n- **H1 (accuracy and native calibration disagree).** The ranking of the three models by RMSE differs from their ranking by native CE on at least one subset. Expected from the pilot: gradient boosting is the most accurate model, and its native quantile intervals under-cover.\n- **H2 (conformal equalises coverage).** After split conformal, every model's PICP at L = 0.80, 0.90 and 0.95 lies within ±0.05 of L on both subsets.\n- **H3 (under conformal, accuracy decides width).** After split conformal, the ranking of models by MPIW at each highlighted level equals their ranking by MAE.\n- **H4 (exploratory).** After split conformal, coverage differs between models within true-RUL bands even when marginal coverage is similar.\n\nA hypothesis is reported as supported, not supported, or inconclusive (when the 95% intervals do not separate the models).\n\n## Deviations\n\n(None at the time of writing.)\n")
open('/content/RUL_project/main/DEVIATIONS.md', 'w', encoding='utf-8').write('# Deviations from the pre-registration\n\n`PREREGISTRATION.md` is kept unchanged so that its recorded SHA-256 stays valid; deviations are listed here.\nAll were found by an independent code review carried out after the first analysis of the main-study results (16 Nov 2026).\n\n| # | What the pre-registration said | What was done | Why | Effect |\n|---|---|---|---|---|\n| D1 | Native gradient-boosting intervals from LightGBM quantile models with early stopping | Native quantiles from scikit-learn `HistGradientBoostingRegressor` (quantile loss; learning rate 0.03, 31 leaves, min 50 samples per leaf, fixed 500 iterations) | With LightGBM, the q = 0.75 model stopped after one tree and predicted a constant 125 for every input in all 10 runs; after the pre-registered sorting this constant became the upper bound of every interval at 80% and above. Boosting starts from the q-quantile of the labels, which is the cap (30% of FD001 and 43% of FD004 training windows have the label 125). A fixed number of trees did not help. The replacement was chosen on validation engines only. | Changes all native gradient-boosting results. Pre-registered (degenerate) outputs kept in `results/raw_prereg_v1/` and `results/tables_prereg_v1/` |\n| D2 | "Inconclusive when the 95% intervals do not separate the models" | Decision rules implemented explicitly (`inference.py`): a ranking claim is decided by paired engine-bootstrap intervals of the differences on **both** rankings involved (supported / not supported only if the relevant pairs are resolved; otherwise inconclusive). H4 tested by paired band-coverage differences at 90% | The first analysis script decided H1–H3 from point estimates and checked separation only for interval width | Changed two pre-registered outcomes after the results were known: H1 on FD004 (supported → inconclusive) and H3 on FD001 (mixed → inconclusive). The point-estimate version is kept in `tables/hypotheses_point_estimate_rule_v1.csv` |\n| D3 | Engine bootstrap of seed-averaged values | Kept as primary; added (a) bootstrap CIs for calibration error, (b) paired RMSE differences, (c) a two-level bootstrap over seeds and engines as sensitivity, (d) H3 with unclipped MAE as sensitivity | Calibration error had no interval; the engine bootstrap ignores training and calibration-split variability | No hypothesis outcome changes under (c) or (d) |\n| D4 | — | LSTM log-variance clamped to [−10, 6] | Numerical safety; an implementation detail not written down | Bounds the predicted s.d. at 0.84–2,500 cycles; never binding in practice |\n| D5 | — | Figure styling and titles in `analyze_main.py` edited after the first results; one figure title that stated a conclusion was replaced by a neutral one | Readability; reviewer request | No metric changed |\n| D6 | Crossing quantiles sorted | Kept; crossing rate now logged | Separate quantile models with 12 closely spaced levels cross often | At least one crossing pair among the 12 quantiles in 85–94% of test engines (per run); sorting is the pre-registered repair |\n\n## D7 — Exploratory addition: post-hoc recalibration (added 19 Nov 2026, after the main results were known)\n\n**Status: exploratory, not pre-registered.** H1–H4 and every number in the main analysis are unchanged.\n\n**Why.** Model-based post-hoc recalibration is the main alternative to conformal prediction for repairing miscalibrated RUL intervals in the prognostics literature (isotonic recalibration of Kuleshov et al., 2018; its use for Bayesian RUL models by Lin & Li, 2022; variance scaling of Levi et al., 2022). Without it a reader cannot tell whether the miscalibration of the native gradient-boosting intervals could have been fixed more cheaply.\n\n**Plan, fixed before running (`run_recal.py`, `analyze_recal.py`).**\n1. Re-fit the three models with the same seeds and splits as the main study; abort if the re-fitted test-set point predictions differ from the saved ones (max abs. difference > 1e-3 cycles).\n2. On the same calibration engines used for conformal, compute the probability integral transform PIT_i = F_i(y_i) of every calibration window under each model\'s native predictive distribution: Gaussian N(mu, sigma^2) for Ridge (sigma = validation RMSE) and the LSTM; for gradient boosting the piecewise-linear CDF through its 12 sorted quantiles, extended linearly to tau = 0 and tau = 1 with the slopes of the two outermost quantile pairs.\n3. **Isotonic recalibration (primary exploratory method).** The recalibration map is the engine-weighted empirical CDF of the PIT values (the isotonic fit of Kuleshov et al. on its own PIT values). The recalibrated central interval at level L is [F^-1(tau_lo), F^-1(tau_hi)] with tau_lo, tau_hi the engine-weighted (1-L)/2 and (1+L)/2 quantiles of PIT. No finite-sample correction.\n4. **Variance scaling (secondary; Ridge and LSTM only, because gradient boosting has no variance parameter).** sigma is multiplied by s, where s^2 is the engine-weighted mean of the squared standardised calibration residuals (the maximum-likelihood scale).\n5. Evaluation exactly as in the main study: test engines, clipping to [0, 125], PICP/MPIW/IS at 50–95%, calibration error CE over the six levels, seed-averaged per-engine values, 2,000-sample engine bootstrap; paired engine-bootstrap differences in CE between recalibrated and native, and between recalibrated and conformal intervals, per model; coverage at 90% by true-RUL band.\n6. Reporting: all results labelled exploratory; no new hypothesis is claimed as confirmed.\n\n**Effect.** Adds `results/raw_recal/`, `results/tables/recal_*.csv` and `results/figures/fig5_recalibration.*`; Section 7.8 of the write-up.\n\n**Post-plan diagnostic (added after the D7 results were seen).** `analyze_recal.py` also writes `recal_lstm_tail_diagnostic.csv`, which compares the LSTM\'s maximum-likelihood variance-scaling factor with the factor needed for 90% coverage on the calibration windows. It explains a result; it changes none.\n\n## D8 — Exploratory addition: epistemic uncertainty and locally adaptive conformal (added 21 Nov 2026, after the main and D7 results were known)\n\n**Status: exploratory, not pre-registered.** H1–H4 and every number in the main analysis and in D7 are unchanged. The plan below was written before any D8 code was run.\n\n**Why.** All native uncertainty in the main study is aleatoric (a noise model fitted to residuals). Two questions a reviewer will ask are therefore open: (a) is the accuracy–calibration gap a property of accurate models, or of the uncertainty mechanism attached to them? (b) Marginal split conformal repairs coverage on average but not where gradient boosting fails (near failure, one-sided); does a locally adaptive conformal method do better?\n\n**Plan, fixed before running (`run_uq.py`, `analyze_uq.py`).**\n1. Same subsets, seeds, splits, windows, features and test engines as the main study. Abort if the re-fitted single LSTM (member 0) or gradient-boosting point predictions differ from the saved ones by more than 1e-3 cycles.\n2. **Deep ensemble** (Lakshminarayanan et al., 2017): M = 5 LSTMs with the main-study architecture, loss (beta-NLL, beta = 0.5), optimiser and early stopping; member k uses torch seed 1000 + seed + 100k (k = 0 is the main-study LSTM). Predictive mean = mean of member means; predictive variance = mean of member variances (aleatoric) + variance of member means (epistemic). Native interval: Gaussian with that mean and variance.\n3. **MC dropout** (Gal & Ghahramani, 2016): the main-study LSTM (trained with dropout p = 0.1 between its two LSTM layers) evaluated with dropout active, T = 50 stochastic passes (torch seed fixed per run). Predictive mean and variance as for the ensemble. No retraining, so the comparison with the single LSTM isolates the inference method.\n4. **Locally adaptive conformal** on the calibration engines, engine-weighted, with the same finite-sample correction as the main study:\n   - gradient boosting: conformalized quantile regression (CQR; Romano et al., 2019) on the native quantiles, score E = max(q_lo(x) − y, y − q_hi(x)), interval [q_lo − Q, q_hi + Q];\n   - LSTM, ensemble and MC dropout: normalised conformal, score |y − mu| / sigma, interval mu ± Q·sigma.\n   Standard (absolute-residual) split conformal is also computed for the ensemble and MC-dropout means.\n5. Evaluation exactly as in the main study (clipping to [0, 125], PICP/MPIW/IS at 50–95%, CE over six levels, seed-averaged per-engine values, 2,000-sample engine bootstrap; coverage at 90% by true-RUL band).\n6. Pre-specified comparisons (paired engine bootstrap): (i) native CE: ensemble vs single LSTM, MC dropout vs single LSTM, ensemble vs gradient boosting; (ii) RMSE: ensemble vs single LSTM, ensemble vs gradient boosting; (iii) adaptive vs standard conformal per model: CE, MPIW and IS at 80/90/95%, and coverage at 90% for true RUL <= 40.\n7. Reporting: all D8 results labelled exploratory; no hypothesis is claimed as confirmed; a resolved difference means a 95% interval excluding zero.\n\n**Effect.** Adds `results/raw_uq/`, `results/tables/uq_*.csv`, `results/figures/fig6_uq.*`; Section 7.9 of the write-up.\n')
print('written')

In [ ]:
%%writefile /content/RUL_project/main/run_main.py
"""Main study (see PREREGISTRATION.md). One call = one (subset, seed).

Saves, for the official test set (last window per engine):
  point predictions, native interval bounds at every nominal level, and the
  split-conformal half-width q_L for every level, for Ridge, LightGBM and LSTM.
No metric is computed here.

Usage: python run_main.py FD001 0
"""
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from scipy.stats import norm

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from data import load, regime_stats, normalise, windows, RUL_CAP  # noqa: E402
from models import handcrafted, beta_nll  # noqa: E402
import conformal as C  # noqa: E402

torch.set_num_threads(2)
LEVELS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
QUANTS = [0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.75, 0.80, 0.85, 0.90, 0.95, 0.975]
OUT = Path(__file__).resolve().parent / "results" / "raw"
SCALE = float(RUL_CAP)


# ---------------------------------------------------------------- splits
def split_engines(units, seed):
    rng = np.random.default_rng(seed)
    u = rng.permutation(np.unique(units))
    n_tr = int(round(0.70 * len(u)))
    train, cal = u[:n_tr], u[n_tr:]
    n_val = max(3, int(round(0.15 * len(train))))
    val, fit = train[:n_val], train[n_val:]
    assert not (set(fit) & set(val)) and not (set(train) & set(cal))
    return fit, val, cal


# ---------------------------------------------------------------- LSTM
class LSTMNet(nn.Module):
    def __init__(self, n_feat, hidden=32, p=0.1):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, num_layers=2, batch_first=True, dropout=p)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU())
        self.mu, self.logvar = nn.Linear(32, 1), nn.Linear(32, 1)

    def forward(self, x):
        h, _ = self.lstm(x)
        z = self.head(h[:, -1])
        return self.mu(z).squeeze(-1), self.logvar(z).squeeze(-1).clamp(-10, 6)


def train_lstm(Xf, yf, Xv, yv, seed, max_epochs=60, patience=8, bs=256):
    torch.manual_seed(seed); np.random.seed(seed)
    m = LSTMNet(Xf.shape[-1])
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    Xf_t, yf_t, Xv_t = torch.from_numpy(Xf), torch.from_numpy(yf / SCALE), torch.from_numpy(Xv)
    g = torch.Generator().manual_seed(seed)
    best, state, bad, ep = np.inf, None, 0, 0
    for ep in range(max_epochs):
        m.train()
        perm = torch.randperm(len(Xf_t), generator=g)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            mu, lv = m(Xf_t[b])
            loss = beta_nll(mu, lv, yf_t[b], 0.5)
            opt.zero_grad(); loss.backward(); opt.step()
        mu, var = predict_lstm(m, Xv)
        nll = float(np.mean(0.5 * np.log(2 * np.pi * var) + 0.5 * (yv - mu) ** 2 / var))
        if nll < best - 1e-4:
            best, bad = nll, 0
            state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    m.load_state_dict(state)
    return m, {"val_nll": best, "epochs": ep + 1}


@torch.no_grad()
def predict_lstm(m, X, bs=4096):
    m.eval()
    mus, vs = [], []
    for i in range(0, len(X), bs):
        mu, lv = m(torch.from_numpy(X[i:i + bs]))
        mus.append(mu.numpy() * SCALE); vs.append(lv.exp().numpy() * SCALE ** 2)
    return np.concatenate(mus), np.concatenate(vs)


# ---------------------------------------------------------------- Ridge / LightGBM
def fit_ridge(Ff, yf, Fv, yv):
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(Ff)
    best = None
    for a in (0.01, 0.1, 1, 10, 100, 1000):
        r = Ridge(alpha=a).fit(sc.transform(Ff), yf)
        mse = float(np.mean((r.predict(sc.transform(Fv)) - yv) ** 2))
        if best is None or mse < best[0]:
            best = (mse, a, r)
    mse, a, r = best
    return (lambda F: r.predict(sc.transform(F))), {"alpha": a, "sigma": float(np.sqrt(mse))}


def lgb_params(seed, **kw):
    return dict(n_estimators=2000, learning_rate=0.03, num_leaves=31, min_child_samples=50, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=2, **kw)


def fit_lgbm(Ff, yf, Fv, yv, seed):
    import lightgbm as lgb
    cb = [lgb.early_stopping(50, verbose=False)]
    point = lgb.LGBMRegressor(**lgb_params(seed, objective="l2")).fit(Ff, yf, eval_set=[(Fv, yv)], callbacks=cb)
    # DEVIATION D1 (see DEVIATIONS.md): with LightGBM's quantile objective the upper-quantile models
    # (q >= 0.75) stay a constant 125 for every input, because boosting starts from the q-quantile of the
    # labels, which is the cap (about 30% of labels tie at 125), and never moves. Native quantiles are
    # therefore fitted with scikit-learn's histogram gradient boosting (same algorithm family), with the
    # pre-registered settings where they translate: learning rate 0.03, 31 leaves, min 50 samples per leaf,
    # fixed 500 iterations (no early stopping). Chosen on validation engines only.
    from sklearn.ensemble import HistGradientBoostingRegressor as HGB
    qm = {q: HGB(loss="quantile", quantile=q, learning_rate=0.03, max_iter=500, max_leaf_nodes=31,
                 min_samples_leaf=50, early_stopping=False, random_state=seed).fit(Ff, yf) for q in QUANTS}
    info = {"trees_point": int(point.best_iteration_), "quantile_impl": "sklearn HistGradientBoosting, 500 iter"}

    def quantiles(F):
        Q = np.stack([qm[q].predict(F) for q in QUANTS], 1)
        info.setdefault("crossing_rate", []).append(float(np.mean(np.any(np.diff(Q, axis=1) < 0, axis=1))))
        return np.sort(Q, 1)  # repair crossing (rate logged)
    return point.predict, quantiles, info


# ---------------------------------------------------------------- conformal
def conformal_q(pred_cal, y_cal, units_cal, level):
    """Engine-weighted split-conformal half-width with the +inf atom (finite-sample
    correction (1-a)(1+1/n_engines); returns inf when too few engines). Coverage is
    approximate/conservative for engine-weighted pooling, not exact."""
    R = np.abs(y_cal - pred_cal)
    return float(C._wq_aug(R[None, :], C.unit_weights(units_cal), np.ones(1), level)[0])


# ---------------------------------------------------------------- main
def main(fd, seed):
    t0 = time.time()
    OUT.mkdir(parents=True, exist_ok=True)
    S, T = load(fd, "train"), load(fd, "test")
    fit_u, val_u, cal_u = split_engines(S.unit.values, seed)
    stats = regime_stats(S[S.unit.isin(np.r_[fit_u, val_u])])   # training engines only
    Xs, _ = normalise(S, stats)
    Xt, _ = normalise(T, stats)

    def sub(units):
        m = S.unit.isin(units).values
        return windows(S[m].reset_index(drop=True), Xs[m])

    (Xf, yf, _, _), (Xv, yv, _, _), (Xc, yc, _, uc) = sub(fit_u), sub(val_u), sub(cal_u)
    Xte, yte, yte_raw, ute = windows(T, Xt, last_only=True)
    Ff, Fv, Fc, Fte = map(handcrafted, (Xf, Xv, Xc, Xte))

    out = {"y": yte, "y_raw": yte_raw, "unit": ute}
    meta = {"subset": fd, "seed": seed, "n_fit": len(fit_u), "n_val": len(val_u), "n_cal": len(cal_u),
            "cal_units": cal_u.tolist(), "levels": LEVELS}

    def store(name, pt_te, pt_cal, native):  # native: dict level -> (lo, hi) on test
        out[f"{name}_point"] = pt_te
        for L in LEVELS:
            lo, hi = native[L]
            out[f"{name}_native_lo_{L}"], out[f"{name}_native_hi_{L}"] = lo, hi
            out[f"{name}_cq_{L}"] = np.array(conformal_q(pt_cal, yc, uc, L))
        out[f"{name}_cal_point"] = pt_cal

    # Ridge
    rp, rinfo = fit_ridge(Ff, yf, Fv, yv)
    p_te = rp(Fte)
    store("ridge", p_te, rp(Fc), {L: (p_te - norm.ppf(.5 + L / 2) * rinfo["sigma"],
                                      p_te + norm.ppf(.5 + L / 2) * rinfo["sigma"]) for L in LEVELS})
    meta["ridge"] = rinfo

    # LightGBM
    gp, gq, ginfo = fit_lgbm(Ff, yf, Fv, yv, seed)
    Q = gq(Fte); qi = {q: i for i, q in enumerate(QUANTS)}
    p_te = gp(Fte)
    store("gbm", p_te, gp(Fc), {L: (Q[:, qi[round((1 - L) / 2, 3)]], Q[:, qi[round((1 + L) / 2, 3)]]) for L in LEVELS})
    meta["gbm"] = ginfo

    # LSTM
    net, linfo = train_lstm(Xf, yf, Xv, yv, seed=1000 + seed)
    mu_te, var_te = predict_lstm(net, Xte)
    mu_c, _ = predict_lstm(net, Xc)
    s = np.sqrt(var_te)
    store("lstm", mu_te, mu_c, {L: (mu_te - norm.ppf(.5 + L / 2) * s, mu_te + norm.ppf(.5 + L / 2) * s) for L in LEVELS})
    out["lstm_sd"] = s
    meta["lstm"] = linfo

    # validation-set point error (for reporting model fit; never used for selection beyond pre-registered rules)
    meta["val_rmse"] = {"ridge": float(np.sqrt(np.mean((rp(Fv) - yv) ** 2))),
                        "gbm": float(np.sqrt(np.mean((gp(Fv) - yv) ** 2))),
                        "lstm": float(np.sqrt(np.mean((predict_lstm(net, Xv)[0] - yv) ** 2)))}
    vcov = {}
    pv_r = rp(Fv); Qv = gq(Fv); mv, vv = predict_lstm(net, Xv)
    for L in LEVELS:
        z_ = norm.ppf(.5 + L / 2)
        vcov[str(L)] = {"ridge": float(np.mean(np.abs(yv - pv_r) <= z_ * rinfo["sigma"])),
                        "gbm": float(np.mean((yv >= Qv[:, qi[round((1 - L) / 2, 3)]]) & (yv <= Qv[:, qi[round((1 + L) / 2, 3)]]))),
                        "lstm": float(np.mean(np.abs(yv - mv) <= z_ * np.sqrt(vv)))}
    meta["val_native_coverage_windows"] = vcov
    np.savez_compressed(OUT / f"{fd}_seed{seed}.npz", **out)
    meta["seconds"] = time.time() - t0
    (OUT / f"{fd}_seed{seed}.json").write_text(json.dumps(meta, indent=1, default=float))
    print(f"[{fd} s{seed}] done {meta['seconds']:.0f}s val_rmse={meta['val_rmse']} lstm_ep={linfo['epochs']}", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], int(sys.argv[2]))

In [ ]:
%%writefile /content/RUL_project/main/analyze_main.py
"""Analysis of the main study, exactly as pre-registered (PREREGISTRATION.md).
Writes results/tables/*.csv and results/figures/*.png|pdf.  Usage: python analyze_main.py"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent
RAW, TAB, FIG = ROOT / "results" / "raw", ROOT / "results" / "tables", ROOT / "results" / "figures"
TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
LEVELS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
HIGH = [0.80, 0.90, 0.95]
MODELS = ["ridge", "gbm", "lstm"]
NAME = {"ridge": "Ridge", "gbm": "Gradient boosting", "lstm": "LSTM"}
CAP, SUBSETS = 125.0, ["FD001", "FD004"]
BANDS = [(0, 40), (40, 80), (80, 125)]


def nasa_mean(p, y):
    d = p - y
    return float(np.mean(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))


def interval_score(lo, hi, y, a):
    return (hi - lo) + (2 / a) * (lo - y) * (y < lo) + (2 / a) * (y - hi) * (y > hi)


def intervals(z, m, method, L):
    p = z[f"{m}_point"]
    if method == "native":
        return z[f"{m}_native_lo_{L}"], z[f"{m}_native_hi_{L}"]
    q = float(z[f"{m}_cq_{L}"])
    return p - q, p + q


def collect():
    rows, eng = [], []  # eng: per-engine records for bootstrap
    for f in sorted(RAW.glob("*.npz")):
        fd, seed = f.stem.split("_seed")[0], int(f.stem.split("_seed")[1])
        z = dict(np.load(f)); y, yraw, u = z["y"], z["y_raw"], z["unit"]
        for m in MODELS:
            p = z[f"{m}_point"]; pc = np.clip(p, 0, CAP)
            base = {"subset": fd, "seed": seed, "model": m}
            rows.append({**base, "method": "point", "level": np.nan, "MAE": float(np.mean(np.abs(pc - y))),
                         "RMSE": float(np.sqrt(np.mean((pc - y) ** 2))),
                         "RMSE_unclipped": float(np.sqrt(np.mean((p - y) ** 2))),
                         "NASA_mean_raw": nasa_mean(p, yraw)})
            for e, ae in zip(u, np.abs(pc - y)):
                eng.append({**base, "method": "point", "level": np.nan, "unit": int(e), "abs_err": float(ae), "sq_err": float(ae ** 2)})
            for method in ("native", "conformal"):
                for L in LEVELS:
                    lo, hi = intervals(z, m, method, L)
                    inf = ~np.isfinite(hi)
                    lc, hc = np.clip(lo, 0, CAP), np.clip(hi, 0, CAP)
                    cov = (y >= lc) & (y <= hc); w = hc - lc
                    isc = interval_score(lc, hc, y, 1 - L)
                    unc = y < CAP
                    r = {**base, "method": method, "level": L, "PICP": float(cov.mean()), "MPIW": float(w.mean()),
                         "IS": float(isc.mean()), "frac_infinite": float(inf.mean()),
                         "PICP_unclipped": float(((y >= lo) & (y <= hi)).mean()),
                         "MPIW_unclipped": float(np.mean(hi - lo)) if not inf.any() else np.inf,
                         "PICP_uncapped_pts": float(cov[unc].mean())}
                    for a, b in BANDS:
                        s = (y > a) & (y <= b) if a > 0 else (y <= b)
                        r[f"PICP_band_{a}-{b}"] = float(cov[s].mean()); r[f"n_band_{a}-{b}"] = int(s.sum())
                    rows.append(r)
                    for e, c, ww, ii in zip(u, cov, w, isc):
                        eng.append({**base, "method": method, "level": L, "unit": int(e), "cov": float(c),
                                    "width": float(ww), "is": float(ii)})
    return pd.DataFrame(rows), pd.DataFrame(eng)


def boot_ci(v, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(v); b = v[rng.integers(0, len(v), (n, len(v)))].mean(1)
    return float(np.quantile(b, .025)), float(np.quantile(b, .975))


def summarise(D, E):
    # seed mean and s.d.
    pt = D[D.method == "point"].groupby(["subset", "model"])[["MAE", "RMSE", "RMSE_unclipped", "NASA_mean_raw"]].agg(["mean", "std"])
    pt.to_csv(TAB / "accuracy.csv")
    iv = D[D.method != "point"].groupby(["subset", "model", "method", "level"])[
        ["PICP", "MPIW", "IS", "frac_infinite", "PICP_uncapped_pts", "PICP_band_0-40", "PICP_band_40-80", "PICP_band_80-125"]].agg(["mean", "std"])
    iv.to_csv(TAB / "intervals.csv")
    # calibration error per seed, then mean/sd
    ce = (D[D.method != "point"].assign(abs_dev=lambda d: (d.PICP - d.level).abs())
          .groupby(["subset", "seed", "model", "method"]).abs_dev.mean().rename("CE").reset_index())
    ce.groupby(["subset", "model", "method"]).CE.agg(["mean", "std"]).to_csv(TAB / "calibration_error.csv")

    # engine-level bootstrap CIs: per-engine values averaged over seeds (test engines identical across seeds)
    ci_rows = []
    pe = E[E.method == "point"].groupby(["subset", "model", "unit"])[["abs_err", "sq_err"]].mean().reset_index()
    for (fd, m), g in pe.groupby(["subset", "model"]):
        lo, hi = boot_ci(g.abs_err); ci_rows.append({"subset": fd, "model": m, "metric": "MAE", "level": np.nan, "method": "point",
                                                     "value": g.abs_err.mean(), "lo95": lo, "hi95": hi})
        b = g.sq_err.values[np.random.default_rng(0).integers(0, len(g), (2000, len(g)))].mean(1) ** .5
        ci_rows.append({"subset": fd, "model": m, "metric": "RMSE", "level": np.nan, "method": "point",
                        "value": float(np.sqrt(g.sq_err.mean())), "lo95": float(np.quantile(b, .025)), "hi95": float(np.quantile(b, .975))})
    ie = E[E.method != "point"].groupby(["subset", "model", "method", "level", "unit"])[["cov", "width", "is"]].mean().reset_index()
    for (fd, m, meth, L), g in ie.groupby(["subset", "model", "method", "level"]):
        for met, col in (("PICP", "cov"), ("MPIW", "width"), ("IS", "is")):
            lo, hi = boot_ci(g[col]); ci_rows.append({"subset": fd, "model": m, "metric": met, "level": L, "method": meth,
                                                      "value": g[col].mean(), "lo95": lo, "hi95": hi})
    CI = pd.DataFrame(ci_rows); CI.to_csv(TAB / "bootstrap_ci.csv", index=False)

    # paired model differences (engine bootstrap)
    pair_rows = []
    pairs = [("gbm", "ridge"), ("gbm", "lstm"), ("lstm", "ridge")]
    for fd in SUBSETS:
        for a, b in pairs:
            A = pe[(pe.subset == fd) & (pe.model == a)].set_index("unit"); B = pe[(pe.subset == fd) & (pe.model == b)].set_index("unit")
            d = (A.abs_err - B.abs_err).values; lo, hi = boot_ci(d)
            pair_rows.append({"subset": fd, "A": a, "B": b, "metric": "MAE", "method": "point", "level": np.nan, "diff": d.mean(), "lo95": lo, "hi95": hi})
            for meth in ("native", "conformal"):
                for L in HIGH:
                    sel = (ie.subset == fd) & (ie.method == meth) & (ie.level == L)
                    A = ie[sel & (ie.model == a)].set_index("unit"); B = ie[sel & (ie.model == b)].set_index("unit")
                    for met, col in (("PICP", "cov"), ("MPIW", "width"), ("IS", "is")):
                        d = (A[col] - B[col]).values; lo, hi = boot_ci(d)
                        pair_rows.append({"subset": fd, "A": a, "B": b, "metric": met, "method": meth, "level": L,
                                          "diff": d.mean(), "lo95": lo, "hi95": hi})
    P = pd.DataFrame(pair_rows); P.to_csv(TAB / "paired_differences.csv", index=False)
    return pt, iv, ce, CI, P


def hypotheses(D, ce, P):
    out = []
    acc = D[D.method == "point"].groupby(["subset", "model"]).RMSE.mean()
    mae = D[D.method == "point"].groupby(["subset", "model"]).MAE.mean()
    cem = ce.groupby(["subset", "model", "method"]).CE.mean()
    iv = D[D.method != "point"].groupby(["subset", "model", "method", "level"])[["PICP", "MPIW"]].mean()
    for fd in SUBSETS:
        r_rmse = acc[fd].sort_values().index.tolist()
        r_ce = cem.xs((fd, "native"), level=("subset", "method")).sort_values().index.tolist()
        out.append({"hypothesis": "H1", "subset": fd, "detail": f"RMSE rank {r_rmse}; native CE rank {r_ce}",
                    "result": "supported" if r_rmse != r_ce else "not supported"})
        bad = [(m, L, round(iv.loc[(fd, m, 'conformal', L), 'PICP'], 3)) for m in MODELS for L in HIGH
               if abs(iv.loc[(fd, m, "conformal", L), "PICP"] - L) > 0.05]
        out.append({"hypothesis": "H2", "subset": fd, "detail": f"outside ±0.05: {bad}" if bad else "all within ±0.05",
                    "result": "supported" if not bad else "not supported"})
        r_mae = mae[fd].sort_values().index.tolist()
        for L in HIGH:
            r_w = iv.xs((fd, "conformal", L), level=("subset", "method", "level")).MPIW.sort_values().index.tolist()
            # separation: are the MPIW differences between adjacent models resolved by the paired CI?
            sep = []
            for a, b in zip(r_w[:-1], r_w[1:]):
                row = P[(P.subset == fd) & (P.metric == "MPIW") & (P.method == "conformal") & (P.level == L) &
                        (((P.A == a) & (P.B == b)) | ((P.A == b) & (P.B == a)))]
                sep.append(bool(((row.lo95 > 0) | (row.hi95 < 0)).iloc[0]))
            res = "supported" if r_w == r_mae else "not supported"
            if r_w != r_mae and not all(sep):
                res = "inconclusive"
            out.append({"hypothesis": "H3", "subset": fd, "detail": f"L={L}: MPIW rank {r_w}; MAE rank {r_mae}; adjacent pairs separated: {sep}",
                        "result": res})
    H = pd.DataFrame(out); H.to_csv(TAB / "hypotheses_point_estimate_rule_v1.csv", index=False)
    return H


# ---------------------------------------------------------------- figures
INK, INK2, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
COL = {"ridge": "#2a78d6", "gbm": "#eb6834", "lstm": "#1baf7a"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK2,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlecolor": INK, "axes.titlesize": 10,
                     "figure.facecolor": SURF, "axes.facecolor": SURF, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "legend.frameon": False, "savefig.dpi": 200, "savefig.bbox": "tight"})


def save(fig, n):
    fig.savefig(FIG / f"{n}.png"); fig.savefig(FIG / f"{n}.pdf"); plt.close(fig)


def fig_calibration_curves(D, CI):
    fig, axes = plt.subplots(2, 2, figsize=(8.6, 7.4), sharex=True, sharey=True, constrained_layout=True)
    for i, fd in enumerate(SUBSETS):
        for j, meth in enumerate(("native", "conformal")):
            ax = axes[i, j]; ax.plot([0.45, 1], [0.45, 1], color=MUTED, lw=1, ls="--")
            for m in MODELS:
                g = CI[(CI.subset == fd) & (CI.model == m) & (CI.method == meth) & (CI.metric == "PICP")].sort_values("level")
                ax.fill_between(g.level, g.lo95, g.hi95, color=COL[m], alpha=0.15, lw=0)
                ax.plot(g.level, g.value, color=COL[m], lw=2, marker="o", ms=4.5, label=NAME[m])
            ax.set_title(f"{fd} — {'native intervals' if meth == 'native' else 'split conformal'}")
            ax.set_xlim(0.45, 1.0); ax.set_ylim(0.2, 1.0)
            if i == 1: ax.set_xlabel("nominal coverage")
            if j == 0: ax.set_ylabel("empirical coverage (test engines)")
    h, l = axes[0, 0].get_legend_handles_labels(); fig.legend(h, l, loc="outside lower center", ncol=3)
    fig.suptitle("Calibration curves: dashed line = perfect calibration; bands = 95% engine bootstrap", color=INK)
    save(fig, "fig1_calibration_curves")


def fig_accuracy_vs_calibration(D, ce):
    acc = D[D.method == "point"][["subset", "seed", "model", "RMSE"]]
    M = acc.merge(ce, on=["subset", "seed", "model"])
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), constrained_layout=True)
    for ax, fd in zip(axes, SUBSETS):
        g = M[M.subset == fd]
        for m in MODELS:
            for meth, mk in (("native", "o"), ("conformal", "s")):
                h = g[(g.model == m) & (g.method == meth)]
                if meth == "native":
                    ax.scatter(h.RMSE, h.CE, marker=mk, s=30, facecolor=COL[m], edgecolor=SURF, lw=1, zorder=3)
                    ax.scatter(h.RMSE.mean(), h.CE.mean(), marker=mk, s=120, facecolor=COL[m], edgecolor=INK, lw=1, zorder=4)
                else:
                    ax.scatter(h.RMSE, h.CE, marker=mk, s=26, facecolor="none", edgecolor=COL[m], lw=1, zorder=3)
                    ax.scatter(h.RMSE.mean(), h.CE.mean(), marker=mk, s=120, facecolor=SURF, edgecolor=COL[m], lw=2.2, zorder=4)
        ax.set_title(fd); ax.set_xlabel("RMSE (cycles) — lower is more accurate")
    axes[0].set_ylabel("calibration error  mean|PICP − nominal|")
    from matplotlib.lines import Line2D
    hs = [Line2D([], [], color=COL[m], marker="o", ls="", ms=7, label=NAME[m]) for m in MODELS] + \
         [Line2D([], [], color=INK2, marker="o", ls="", ms=7, label="native"),
          Line2D([], [], color=INK2, marker="s", ls="", ms=7, mfc="none", label="split conformal")]
    fig.legend(handles=hs, loc="outside lower center", ncol=5, fontsize=8)
    fig.suptitle("Point accuracy vs calibration error (small marks: seeds; large: mean over seeds)", color=INK)
    save(fig, "fig3_accuracy_vs_calibration")


def fig_width(D):
    iv = D[D.method != "point"].groupby(["subset", "model", "method", "level"])[["MPIW", "PICP"]].mean().reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True, constrained_layout=True)
    for ax, fd in zip(axes, SUBSETS):
        x0 = np.arange(len(HIGH))
        for k, m in enumerate(MODELS):
            for meth, alpha, off in (("native", 0.45, -0.07), ("conformal", 1.0, 0.07)):
                g = iv[(iv.subset == fd) & (iv.model == m) & (iv.method == meth) & iv.level.isin(HIGH)].sort_values("level")
                ax.bar(x0 + (k - 1) * 0.27 + off, g.MPIW, 0.13, color=COL[m], alpha=alpha,
                       label=f"{NAME[m]} ({meth})" if fd == "FD001" else None)
        ax.set_xticks(x0, [f"{int(L*100)}%" for L in HIGH]); ax.set_title(fd); ax.set_xlabel("nominal level")
    axes[0].set_ylabel("mean interval width (cycles)")
    fig.legend(loc="outside lower center", ncol=3, fontsize=7.5)
    fig.suptitle("Interval width: native (light) vs split conformal (solid)", color=INK)
    save(fig, "fig2_width")


def fig_bands(D):
    cols = ["PICP_band_0-40", "PICP_band_40-80", "PICP_band_80-125"]
    iv = D[(D.method != "point") & (D.level == 0.90)].groupby(["subset", "model", "method"])[cols].mean()
    fig, axes = plt.subplots(2, 2, figsize=(8.6, 6), sharey=True, constrained_layout=True)
    for i, fd in enumerate(SUBSETS):
        for j, meth in enumerate(("native", "conformal")):
            ax = axes[i, j]
            for k, m in enumerate(MODELS):
                ax.bar(np.arange(3) + (k - 1) * 0.26, iv.loc[(fd, m, meth)].values, 0.24, color=COL[m], label=NAME[m])
            ax.axhline(0.9, color=MUTED, lw=1, ls="--"); ax.set_ylim(0, 1.05)
            nb = D[(D.subset == fd) & (D.method != "point")].iloc[0]
            ax.set_xticks(range(3), [f"RUL ≤ 40\n(n={int(nb['n_band_0-40'])})", f"40–80\n(n={int(nb['n_band_40-80'])})",
                                     f"80–125\n(n={int(nb['n_band_80-125'])})"]); ax.set_title(f"{fd} — {meth}")
    axes[0, 0].set_ylabel("coverage at nominal 90%"); axes[1, 0].set_ylabel("coverage at nominal 90%")
    h, l = axes[0, 0].get_legend_handles_labels(); fig.legend(h, l, loc="outside lower center", ncol=3)
    fig.suptitle("Coverage by true remaining life (exploratory, H4)", color=INK)
    save(fig, "fig4_bands")


if __name__ == "__main__":
    D, E = collect()
    D.to_csv(TAB / "metrics_per_seed.csv", index=False)
    pt, iv, ce, CI, P = summarise(D, E)
    H = hypotheses(D, ce, P)
    fig_calibration_curves(D, CI); fig_accuracy_vs_calibration(D, ce); fig_width(D); fig_bands(D)
    print(H.to_string(index=False))
    print("done")

In [ ]:
%%writefile /content/RUL_project/main/inference.py
"""Hypothesis tests with explicit decision rules (deviation D2, see DEVIATIONS.md).

Pre-registered wording: a hypothesis is 'inconclusive when the 95% intervals do not separate
the models'. The first analysis script decided H1-H3 from point estimates. Here every decision
uses paired bootstrap intervals of the model differences on BOTH rankings involved.

Primary bootstrap (pre-registered): resample test engines (2,000x) using per-engine values
averaged over the 5 seeds. Sensitivity: two-level bootstrap resampling seeds and engines.
"""
from __future__ import annotations

from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent
RAW, TAB = ROOT / "results" / "raw", ROOT / "results" / "tables"
LEVELS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
HIGH = [0.80, 0.90, 0.95]
MODELS = ["ridge", "gbm", "lstm"]
SUBSETS = ["FD001", "FD004"]
CAP, B = 125.0, 2000
BANDS = [(0, 40), (40, 80), (80, 125)]


def load_arrays(fd):
    """A[(model, kind)] -> array (seeds, engines). kinds: abs, sq, abs_raw, ('cov'|'w', method, L)."""
    A, y = {}, None
    files = sorted(RAW.glob(f"{fd}_seed*.npz"))
    for f in files:
        z = dict(np.load(f)); y = z["y"]
        for m in MODELS:
            p = z[f"{m}_point"]; pc = np.clip(p, 0, CAP)
            A.setdefault((m, "abs"), []).append(np.abs(pc - y))
            A.setdefault((m, "sq"), []).append((pc - y) ** 2)
            A.setdefault((m, "abs_raw"), []).append(np.abs(p - y))
            for meth in ("native", "conformal"):
                for L in LEVELS:
                    if meth == "native":
                        lo, hi = z[f"{m}_native_lo_{L}"], z[f"{m}_native_hi_{L}"]
                    else:
                        q = float(z[f"{m}_cq_{L}"]); lo, hi = p - q, p + q
                    lc, hc = np.clip(lo, 0, CAP), np.clip(hi, 0, CAP)
                    A.setdefault((m, "cov", meth, L), []).append(((y >= lc) & (y <= hc)).astype(float))
                    A.setdefault((m, "w", meth, L), []).append(hc - lc)
    return {k: np.stack(v) for k, v in A.items()}, y


def stat(A, m, what, idx_s, idx_e, meth=None, L=None, mask=None):
    """Statistic on a (seed, engine) resample. idx_s: seed indices; idx_e: engine indices."""
    def take(k):
        X = A[k][idx_s][:, idx_e]
        if mask is not None:
            X = X[:, mask[idx_e]]
        return X
    if what == "MAE":
        return take((m, "abs")).mean()
    if what == "MAE_raw":
        return take((m, "abs_raw")).mean()
    if what == "RMSE":
        return np.sqrt(take((m, "sq")).mean())
    if what == "PICP":
        return take((m, "cov", meth, L)).mean()
    if what == "MPIW":
        return take((m, "w", meth, L)).mean()
    if what == "CE":
        return np.mean([abs(take((m, "cov", meth, l)).mean() - l) for l in LEVELS])
    raise ValueError(what)


def boot(A, n_s, n_e, fn, two_level, seed):
    rng = np.random.default_rng(seed)
    out = np.empty(B)
    all_s = np.arange(n_s)
    for b in range(B):
        idx_s = rng.integers(0, n_s, n_s) if two_level else all_s
        idx_e = rng.integers(0, n_e, n_e)
        out[b] = fn(idx_s, idx_e)
    return out


def diff_ci(A, n_s, n_e, a, b, what, meth=None, L=None, mask=None, two_level=False, seed=0):
    f = lambda s, e: stat(A, a, what, s, e, meth, L, mask) - stat(A, b, what, s, e, meth, L, mask)
    full = f(np.arange(n_s), np.arange(n_e))
    bs = boot(A, n_s, n_e, f, two_level, seed)
    lo, hi = np.quantile(bs, [.025, .975])
    return float(full), float(lo), float(hi)


def value_ci(A, n_s, n_e, m, what, meth=None, L=None, two_level=False, seed=0):
    f = lambda s, e: stat(A, m, what, s, e, meth, L)
    bs = boot(A, n_s, n_e, f, two_level, seed)
    return float(f(np.arange(n_s), np.arange(n_e))), *map(float, np.quantile(bs, [.025, .975]))


def resolved(lo, hi):
    return lo > 0 or hi < 0


def decide_ranking(pairs_x, pairs_y):
    """pairs_*: {(a,b): (diff, lo, hi)} for 'lower is better' metrics x and y.
    supported-as-reversal if some pair has resolved opposite signs; 'agree' if every pair is resolved
    on both and signs agree; otherwise inconclusive."""
    reversal = any(resolved(*pairs_x[k][1:]) and resolved(*pairs_y[k][1:]) and
                   np.sign(pairs_x[k][0]) != np.sign(pairs_y[k][0]) for k in pairs_x)
    agree = all(resolved(*pairs_x[k][1:]) and resolved(*pairs_y[k][1:]) and
                np.sign(pairs_x[k][0]) == np.sign(pairs_y[k][0]) for k in pairs_x)
    return "reversal" if reversal else ("agree" if agree else "unresolved")


def run(two_level):
    rows, tests = [], []
    for fd in SUBSETS:
        A, y = load_arrays(fd)
        n_s, n_e = A[(MODELS[0], "abs")].shape
        sd = 0 if not two_level else 1
        pairs = list(combinations(MODELS, 2))
        # value CIs
        for m in MODELS:
            for what in ("MAE", "RMSE"):
                rows.append({"subset": fd, "model": m, "metric": what, "method": "point", "level": np.nan,
                             **dict(zip(("value", "lo95", "hi95"), value_ci(A, n_s, n_e, m, what, two_level=two_level, seed=sd)))})
            for meth in ("native", "conformal"):
                rows.append({"subset": fd, "model": m, "metric": "CE", "method": meth, "level": np.nan,
                             **dict(zip(("value", "lo95", "hi95"), value_ci(A, n_s, n_e, m, "CE", meth, two_level=two_level, seed=sd)))})
        # H1: RMSE vs native CE
        px = {p: diff_ci(A, n_s, n_e, *p, "RMSE", two_level=two_level, seed=sd) for p in pairs}
        py = {p: diff_ci(A, n_s, n_e, *p, "CE", "native", two_level=two_level, seed=sd) for p in pairs}
        d = decide_ranking(px, py)
        tests.append({"hypothesis": "H1", "subset": fd, "level": np.nan,
                      "result": {"reversal": "supported", "agree": "not supported", "unresolved": "inconclusive"}[d],
                      "detail": "; ".join(f"{a}-{b}: dRMSE {px[(a,b)][0]:+.2f} [{px[(a,b)][1]:+.2f},{px[(a,b)][2]:+.2f}], "
                                          f"dCE {py[(a,b)][0]:+.3f} [{py[(a,b)][1]:+.3f},{py[(a,b)][2]:+.3f}]" for a, b in pairs)})
        # H2: conformal PICP within +-0.05 (point estimate, as pre-registered) + CIs reported
        for L in HIGH:
            parts, ok = [], True
            for m in MODELS:
                v, lo, hi = value_ci(A, n_s, n_e, m, "PICP", "conformal", L, two_level=two_level, seed=sd)
                ok &= abs(v - L) <= 0.05
                parts.append(f"{m} {v:.3f} [{lo:.3f},{hi:.3f}]")
            tests.append({"hypothesis": "H2", "subset": fd, "level": L, "result": "supported" if ok else "not supported",
                          "detail": "; ".join(parts)})
        # H3: conformal MPIW vs MAE (clipped, primary) and vs MAE_raw (sensitivity)
        for mae_kind in ("MAE", "MAE_raw"):
            px = {p: diff_ci(A, n_s, n_e, *p, mae_kind, two_level=two_level, seed=sd) for p in pairs}
            for L in HIGH:
                py = {p: diff_ci(A, n_s, n_e, *p, "MPIW", "conformal", L, two_level=two_level, seed=sd) for p in pairs}
                d = decide_ranking(px, py)
                tests.append({"hypothesis": "H3" if mae_kind == "MAE" else "H3 (sens.: unclipped MAE)", "subset": fd, "level": L,
                              "result": {"reversal": "not supported", "agree": "supported", "unresolved": "inconclusive"}[d],
                              "detail": "; ".join(f"{a}-{b}: d{mae_kind} {px[(a,b)][0]:+.2f} [{px[(a,b)][1]:+.2f},{px[(a,b)][2]:+.2f}], "
                                                  f"dMPIW {py[(a,b)][0]:+.2f} [{py[(a,b)][1]:+.2f},{py[(a,b)][2]:+.2f}]" for a, b in pairs)})
        # H4: conformal 90% coverage within true-RUL bands, paired model differences
        res, parts = False, []
        for lo_b, hi_b in BANDS:
            mask = (y > lo_b) & (y <= hi_b) if lo_b > 0 else (y <= hi_b)
            for a, b in pairs:
                dd = diff_ci(A, n_s, n_e, a, b, "PICP", "conformal", 0.90, mask=mask, two_level=two_level, seed=sd)
                if resolved(*dd[1:]):
                    res = True
                parts.append(f"band {lo_b}-{hi_b} (n={int(mask.sum())}) {a}-{b}: {dd[0]:+.3f} [{dd[1]:+.3f},{dd[2]:+.3f}]")
        tests.append({"hypothesis": "H4", "subset": fd, "level": 0.90, "result": "supported" if res else "inconclusive",
                      "detail": "; ".join(parts)})
    return pd.DataFrame(rows), pd.DataFrame(tests)


if __name__ == "__main__":
    V, T = run(two_level=False)
    V.to_csv(TAB / "value_ci_primary.csv", index=False); T.to_csv(TAB / "hypotheses_primary.csv", index=False)
    V2, T2 = run(two_level=True)
    V2.to_csv(TAB / "value_ci_two_level.csv", index=False); T2.to_csv(TAB / "hypotheses_two_level.csv", index=False)
    M = T[["hypothesis", "subset", "level", "result"]].merge(
        T2[["hypothesis", "subset", "level", "result"]], on=["hypothesis", "subset", "level"], suffixes=("_primary", "_two_level"))
    print(M.to_string(index=False))

In [ ]:
%%writefile /content/RUL_project/main/extra_checks.py
"""Post-review supporting analyses (descriptive): MAE and native width by true-RUL band,
paired native near-failure coverage CI, calibration-engine coverage of conformal q,
and chosen hyperparameters per run. Writes results/tables/extra_*.csv"""
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
sys.argv = ["x"]
from run_main import split_engines, LEVELS
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from data import load, regime_stats, normalise, windows
import conformal as C
from inference import load_arrays, diff_ci

R = Path(__file__).resolve().parent / "results"; T = R / "tables"
BANDS = [(0, 40), (40, 80), (80, 125)]
rows, cal_rows, hp = [], [], []
for fd in ("FD001", "FD004"):
    for f in sorted((R / "raw").glob(f"{fd}_seed*.npz")):
        seed = int(f.stem.split("seed")[1]); z = dict(np.load(f)); y = z["y"]
        meta = json.loads(f.with_suffix(".json").read_text())
        hp.append({"subset": fd, "seed": seed, "ridge_alpha": meta["ridge"]["alpha"], "ridge_sigma": meta["ridge"]["sigma"],
                   "gbm_point_trees": meta["gbm"]["trees_point"], "lstm_epochs": meta["lstm"]["epochs"],
                   "ridge_frac_pred_above_125": float(np.mean(z["ridge_point"] > 125))})
        for m in ("ridge", "gbm", "lstm"):
            p = np.clip(z[f"{m}_point"], 0, 125)
            for a, b in BANDS:
                s = (y > a) & (y <= b) if a > 0 else (y <= b)
                w = np.clip(z[f"{m}_native_hi_0.9"], 0, 125) - np.clip(z[f"{m}_native_lo_0.9"], 0, 125)
                rows.append({"subset": fd, "seed": seed, "model": m, "band": f"{a}-{b}", "MAE": float(np.mean(np.abs(p - y)[s])),
                             "native_width_90": float(w[s].mean())})
        # calibration-engine coverage of the conformal half-width (reconstruct calibration labels deterministically)
        S = load(fd, "train"); fit_u, val_u, cal_u = split_engines(S.unit.values, seed)
        st = regime_stats(S[S.unit.isin(np.r_[fit_u, val_u])]); Xs, _ = normalise(S, st)
        msk = S.unit.isin(cal_u).values
        _, yc, _, uc = windows(S[msk].reset_index(drop=True), Xs[msk])
        wts = C.unit_weights(uc); wts = wts / wts.sum()
        for m in ("ridge", "gbm", "lstm"):
            pc = z[f"{m}_cal_point"]; assert len(pc) == len(yc)
            for L in LEVELS:
                q = float(z[f"{m}_cq_{L}"])
                cal_rows.append({"subset": fd, "seed": seed, "model": m, "level": L,
                                 "cal_engine_weighted_coverage": float(wts @ (np.abs(yc - pc) <= q)),
                                 "target": min(1.0, L * (1 + 1 / len(cal_u)))})
B = pd.DataFrame(rows).groupby(["subset", "model", "band"])[["MAE", "native_width_90"]].mean().reset_index()
B.to_csv(T / "extra_by_band.csv", index=False)
CC = pd.DataFrame(cal_rows).groupby(["subset", "model", "level"])[["cal_engine_weighted_coverage", "target"]].mean().reset_index()
CC.to_csv(T / "extra_calibration_engine_coverage.csv", index=False)
H = pd.DataFrame(hp); H.to_csv(T / "extra_hyperparameters.csv", index=False)
# paired native near-failure coverage (90%) GBM vs Ridge and vs LSTM
pr = []
for fd in ("FD001", "FD004"):
    A, y = load_arrays(fd); n_s, n_e = A[("ridge", "abs")].shape
    for lo, hi in BANDS:
        mask = (y > lo) & (y <= hi) if lo > 0 else (y <= hi)
        for a, b in (("gbm", "ridge"), ("gbm", "lstm"), ("lstm", "ridge")):
            d = diff_ci(A, n_s, n_e, a, b, "PICP", "native", 0.90, mask=mask)
            pr.append({"subset": fd, "band": f"{lo}-{hi}", "A": a, "B": b, "diff": d[0], "lo95": d[1], "hi95": d[2]})
pd.DataFrame(pr).to_csv(T / "extra_native_band_paired.csv", index=False)
pd.set_option("display.width", 200)
print(B.round(2).to_string(index=False)); print(CC[CC.level.isin([0.8, 0.9, 0.95])].round(3).to_string(index=False))
print(H.round(2).to_string(index=False)); print(pd.DataFrame(pr).round(3).to_string(index=False))

In [ ]:
%%writefile /content/RUL_project/main/run_recal.py
"""Exploratory deviation D7 (see DEVIATIONS.md): post-hoc recalibration of the native intervals.

Re-fits the three models exactly as run_main.py (same seed, same splits), checks that the test-set
point predictions reproduce the saved main-study outputs, then builds recalibrated intervals from the
calibration engines:
  * isotonic recalibration (Kuleshov et al., 2018): map nominal quantile levels through the
    engine-weighted empirical CDF of the calibration PIT values (all three models);
  * variance scaling (Levi et al., 2022): sigma * s with s^2 = engine-weighted mean squared z (Ridge, LSTM).
Saves results/raw_recal/{FD}_seed{s}.npz. No metric is computed here.

Usage: python run_recal.py FD001 0
"""
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
from scipy.stats import norm

import run_main as RM
from run_main import LEVELS, QUANTS

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from data import load, regime_stats, normalise, windows  # noqa: E402
from models import handcrafted  # noqa: E402
import conformal as C  # noqa: E402

ROOT = Path(__file__).resolve().parent
OUT = ROOT / "results" / "raw_recal"
TAUS = np.array(QUANTS)


# ---------------------------------------------------------------- helpers
def wquantile(v, w, q):
    """Weighted quantile (inverse of the weighted empirical CDF, right-continuous step)."""
    o = np.argsort(v)
    v, cw = v[o], np.cumsum(w[o]) / w.sum()
    return v[np.searchsorted(cw, q - 1e-12)]


def gbm_knots(Q):
    """Piecewise-linear quantile function through the sorted quantiles, extended linearly to tau=0 and 1.
    Q: (n, 12) sorted. Returns taus (14,) and values (n, 14), strictly increasing in tau."""
    lo = Q[:, 0] - (Q[:, 1] - Q[:, 0]) / (TAUS[1] - TAUS[0]) * TAUS[0]
    hi = Q[:, -1] + (Q[:, -1] - Q[:, -2]) / (TAUS[-1] - TAUS[-2]) * (1 - TAUS[-1])
    t = np.r_[0.0, TAUS, 1.0]
    V = np.column_stack([lo, Q, hi])
    V = V + t[None, :] * 1e-6  # break ties so the CDF is well defined
    return t, V


def gbm_pit(Q, y):
    t, V = gbm_knots(Q)
    return np.array([np.interp(yi, vi, t) for yi, vi in zip(y, V)])


def gbm_quantile(Q, tau):
    t, V = gbm_knots(Q)
    j = int(np.clip(np.searchsorted(t, tau) - 1, 0, len(t) - 2))
    f = (tau - t[j]) / (t[j + 1] - t[j])
    return V[:, j] + f * (V[:, j + 1] - V[:, j])


# ---------------------------------------------------------------- main
def main(fd, seed):
    t0 = time.time()
    OUT.mkdir(parents=True, exist_ok=True)
    saved = dict(np.load(ROOT / "results" / "raw" / f"{fd}_seed{seed}.npz"))
    S, T = load(fd, "train"), load(fd, "test")
    fit_u, val_u, cal_u = RM.split_engines(S.unit.values, seed)
    stats = regime_stats(S[S.unit.isin(np.r_[fit_u, val_u])])
    Xs, _ = normalise(S, stats)
    Xt, _ = normalise(T, stats)

    def sub(units):
        m = S.unit.isin(units).values
        return windows(S[m].reset_index(drop=True), Xs[m])

    (Xf, yf, _, _), (Xv, yv, _, _), (Xc, yc, _, uc) = sub(fit_u), sub(val_u), sub(cal_u)
    Xte, yte, _, ute = windows(T, Xt, last_only=True)
    Ff, Fv, Fc, Fte = map(handcrafted, (Xf, Xv, Xc, Xte))
    w = C.unit_weights(uc)

    # re-fit (identical calls to run_main.py)
    rp, rinfo = RM.fit_ridge(Ff, yf, Fv, yv)
    gp, gq, _ = RM.fit_lgbm(Ff, yf, Fv, yv, seed)
    net, _ = RM.train_lstm(Xf, yf, Xv, yv, seed=1000 + seed)

    pred = {"ridge": rp(Fte), "gbm": gp(Fte), "lstm": RM.predict_lstm(net, Xte)[0]}
    check = {m: float(np.max(np.abs(pred[m] - saved[f"{m}_point"]))) for m in pred}
    if max(check.values()) > 1e-3:
        raise RuntimeError(f"re-fit does not reproduce the main study: {check}")

    out = {"y": yte, "unit": ute}
    meta = {"subset": fd, "seed": seed, "reproduction_max_abs_diff": check, "n_cal_engines": len(cal_u)}

    # --- Gaussian models: Ridge (constant sigma) and LSTM (heteroscedastic)
    mu_lstm_c, var_lstm_c = RM.predict_lstm(net, Xc)
    mu_lstm_t, var_lstm_t = RM.predict_lstm(net, Xte)
    gauss = {"ridge": (rp(Fc), np.full(len(yc), rinfo["sigma"]), pred["ridge"], np.full(len(yte), rinfo["sigma"])),
             "lstm": (mu_lstm_c, np.sqrt(var_lstm_c), mu_lstm_t, np.sqrt(var_lstm_t))}
    for m, (mc, sc, mt, st) in gauss.items():
        z = (yc - mc) / sc
        pit = norm.cdf(z)
        s = float(np.sqrt(np.sum(w * z ** 2) / w.sum()))
        meta[f"{m}_varscale_s"] = s
        out[f"{m}_cal_pit"] = pit
        for L in LEVELS:
            tl, th = wquantile(pit, w, (1 - L) / 2), wquantile(pit, w, (1 + L) / 2)
            tl, th = np.clip([tl, th], 1e-6, 1 - 1e-6)
            out[f"{m}_iso_lo_{L}"], out[f"{m}_iso_hi_{L}"] = mt + norm.ppf(tl) * st, mt + norm.ppf(th) * st
            out[f"{m}_iso_tau_{L}"] = np.array([tl, th])
            zL = norm.ppf(.5 + L / 2)
            out[f"{m}_vs_lo_{L}"], out[f"{m}_vs_hi_{L}"] = mt - zL * s * st, mt + zL * s * st

    # --- gradient boosting: quantile-based CDF
    Qc, Qt = gq(Fc), gq(Fte)
    pit = gbm_pit(Qc, yc)
    out["gbm_cal_pit"] = pit
    for L in LEVELS:
        tl, th = wquantile(pit, w, (1 - L) / 2), wquantile(pit, w, (1 + L) / 2)
        out[f"gbm_iso_lo_{L}"], out[f"gbm_iso_hi_{L}"] = gbm_quantile(Qt, tl), gbm_quantile(Qt, th)
        out[f"gbm_iso_tau_{L}"] = np.array([tl, th])
        # sanity: at the native taus the knot interpolation must reproduce the native bounds
        if round((1 - L) / 2, 3) in QUANTS:
            nb = saved[f"gbm_native_lo_{L}"]
            meta.setdefault("gbm_native_reproduction", {})[str(L)] = float(
                np.max(np.abs(gbm_quantile(Qt, (1 - L) / 2) - nb)))

    # calibration-set PIT summary (engine-weighted): fraction of PIT in the central 90% band
    for m in ("ridge", "gbm", "lstm"):
        p = out[f"{m}_cal_pit"]
        meta[f"{m}_cal_native_cov90_engine_weighted"] = float(np.sum(w * ((p >= .05) & (p <= .95))) / w.sum())

    np.savez_compressed(OUT / f"{fd}_seed{seed}.npz", **out)
    meta["seconds"] = time.time() - t0
    (OUT / f"{fd}_seed{seed}.json").write_text(json.dumps(meta, indent=1, default=float))
    print(f"[{fd} s{seed}] recal done {meta['seconds']:.0f}s repro={check}", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], int(sys.argv[2]))

In [ ]:
%%writefile /content/RUL_project/main/analyze_recal.py
"""Exploratory deviation D7: evaluation of post-hoc recalibration against native and conformal intervals.
Same evaluation as the main study (clipping to [0,125], seed-averaged per-engine values, engine bootstrap).
Writes results/tables/recal_*.csv and results/figures/fig5_recalibration.*   Usage: python analyze_recal.py"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

import analyze_main as AM  # figure style, interval score
from inference import load_arrays, diff_ci, value_ci, resolved, LEVELS, HIGH, MODELS, SUBSETS, CAP, BANDS

ROOT = Path(__file__).resolve().parent
RAWR, TAB, FIG = ROOT / "results" / "raw_recal", ROOT / "results" / "tables", ROOT / "results" / "figures"
METHODS = {"native": "native", "conformal": "split conformal", "iso": "isotonic recalibration", "vs": "variance scaling"}


def load_all(fd):
    """Main-study arrays plus recalibrated ones. Adds keys (m,'cov'|'w'|'is', meth, L) for meth in iso, vs."""
    A, y = load_arrays(fd)
    main = sorted((ROOT / "results" / "raw").glob(f"{fd}_seed*.npz"))
    rec = sorted(RAWR.glob(f"{fd}_seed*.npz"))
    assert [f.name for f in main] == [f.name for f in rec], "recalibration runs missing"
    taus = []
    for fm, fr in zip(main, rec):
        z, r = dict(np.load(fm)), dict(np.load(fr))
        assert np.array_equal(z["unit"], r["unit"])
        for m in MODELS:
            p = z[f"{m}_point"]
            for meth in ("native", "conformal", "iso", "vs"):
                for L in LEVELS:
                    if meth == "native":
                        lo, hi = z[f"{m}_native_lo_{L}"], z[f"{m}_native_hi_{L}"]
                    elif meth == "conformal":
                        q = float(z[f"{m}_cq_{L}"]); lo, hi = p - q, p + q
                    else:
                        if f"{m}_{meth}_lo_{L}" not in r:
                            continue
                        lo, hi = r[f"{m}_{meth}_lo_{L}"], r[f"{m}_{meth}_hi_{L}"]
                    lc, hc = np.clip(lo, 0, CAP), np.clip(hi, 0, CAP)
                    if meth in ("iso", "vs"):
                        A.setdefault((m, "cov", meth, L), []).append(((y >= lc) & (y <= hc)).astype(float))
                        A.setdefault((m, "w", meth, L), []).append(hc - lc)
                    A.setdefault((m, "is", meth, L), []).append(AM.interval_score(lc, hc, y, 1 - L))
                if meth == "iso":
                    for L in LEVELS:
                        tl, th = r[f"{m}_iso_tau_{L}"]
                        taus.append({"subset": fd, "seed": int(fm.stem.split("seed")[1]), "model": m, "level": L,
                                     "tau_lo": float(tl), "tau_hi": float(th), "nominal_lo": (1 - L) / 2, "nominal_hi": (1 + L) / 2})
    A = {k: (np.stack(v) if isinstance(v, list) else v) for k, v in A.items()}
    return A, y, taus


def is_ci(A, n_s, n_e, m, meth, L, seed=0):
    X = A[(m, "is", meth, L)].mean(0)
    rng = np.random.default_rng(seed)
    b = X[rng.integers(0, n_e, (2000, n_e))].mean(1)
    return float(X.mean()), float(np.quantile(b, .025)), float(np.quantile(b, .975))


def is_diff_ci(A, n_e, m, ma, mb, L, seed=0):
    d = A[(m, "is", ma, L)].mean(0) - A[(m, "is", mb, L)].mean(0)
    rng = np.random.default_rng(seed)
    b = d[rng.integers(0, n_e, (2000, n_e))].mean(1)
    return float(d.mean()), float(np.quantile(b, .025)), float(np.quantile(b, .975))


def method_diff_ci(A, n_s, n_e, m, ma, mb, what, L=None, mask=None, seed=0):
    """Paired (same model, two interval methods) engine-bootstrap difference; reuses inference.diff_ci by aliasing."""
    B = dict(A)
    for l in LEVELS:
        for kind in ("cov", "w"):
            B[("__a", kind, "x", l)] = A[(m, kind, ma, l)]
            B[("__b", kind, "x", l)] = A[(m, kind, mb, l)]
    return diff_ci(B, n_s, n_e, "__a", "__b", what, "x", L, mask=mask, seed=seed)


def run():
    vals, pairs, bands, taus = [], [], [], []
    for fd in SUBSETS:
        A, y, t = load_all(fd); taus += t
        n_s, n_e = A[("ridge", "abs")].shape
        for m in MODELS:
            meths = ["native", "conformal", "iso"] + (["vs"] if m != "gbm" else [])
            for meth in meths:
                v = value_ci(A, n_s, n_e, m, "CE", meth)
                vals.append({"subset": fd, "model": m, "method": meth, "metric": "CE", "level": np.nan,
                             "value": v[0], "lo95": v[1], "hi95": v[2]})
                for L in LEVELS:
                    for what in ("PICP", "MPIW"):
                        v = value_ci(A, n_s, n_e, m, what, meth, L)
                        vals.append({"subset": fd, "model": m, "method": meth, "metric": what, "level": L,
                                     "value": v[0], "lo95": v[1], "hi95": v[2]})
                    v = is_ci(A, n_s, n_e, m, meth, L)
                    vals.append({"subset": fd, "model": m, "method": meth, "metric": "IS", "level": L,
                                 "value": v[0], "lo95": v[1], "hi95": v[2]})
                for lo_b, hi_b in BANDS:
                    mask = (y > lo_b) & (y <= hi_b) if lo_b > 0 else (y <= hi_b)
                    cov = A[(m, "cov", meth, 0.90)][:, mask].mean()
                    bands.append({"subset": fd, "model": m, "method": meth, "band": f"{lo_b}-{hi_b}",
                                  "n": int(mask.sum()), "PICP90": float(cov)})
            comps = [("iso", "native"), ("iso", "conformal")] + ([("vs", "native"), ("vs", "conformal")] if m != "gbm" else [])
            for ma, mb in comps:
                d = method_diff_ci(A, n_s, n_e, m, ma, mb, "CE")
                pairs.append({"subset": fd, "model": m, "A": ma, "B": mb, "metric": "CE", "level": np.nan,
                              "diff": d[0], "lo95": d[1], "hi95": d[2], "resolved": resolved(*d[1:])})
                for L in HIGH:
                    for what in ("PICP", "MPIW"):
                        d = method_diff_ci(A, n_s, n_e, m, ma, mb, what, L)
                        pairs.append({"subset": fd, "model": m, "A": ma, "B": mb, "metric": what, "level": L,
                                      "diff": d[0], "lo95": d[1], "hi95": d[2], "resolved": resolved(*d[1:])})
                    d = is_diff_ci(A, n_e, m, ma, mb, L)
                    pairs.append({"subset": fd, "model": m, "A": ma, "B": mb, "metric": "IS", "level": L,
                                  "diff": d[0], "lo95": d[1], "hi95": d[2], "resolved": resolved(*d[1:])})
            # near-failure coverage at 90%, iso vs native
            mask = y <= 40
            d = method_diff_ci(A, n_s, n_e, m, "iso", "native", "PICP", 0.90, mask=mask)
            pairs.append({"subset": fd, "model": m, "A": "iso", "B": "native", "metric": "PICP_RUL<=40", "level": 0.90,
                          "diff": d[0], "lo95": d[1], "hi95": d[2], "resolved": resolved(*d[1:])})
    V, P, Bd = pd.DataFrame(vals), pd.DataFrame(pairs), pd.DataFrame(bands)
    Tt = pd.DataFrame(taus).groupby(["subset", "model", "level"])[["nominal_lo", "tau_lo", "nominal_hi", "tau_hi"]].mean().reset_index()
    V.to_csv(TAB / "recal_values.csv", index=False); P.to_csv(TAB / "recal_paired.csv", index=False)
    Bd.to_csv(TAB / "recal_bands.csv", index=False); Tt.to_csv(TAB / "recal_taus.csv", index=False)
    meta = [json.loads(f.read_text()) for f in sorted(RAWR.glob("*.json"))]
    M = pd.DataFrame([{"subset": d["subset"], "seed": d["seed"], "repro_max": max(d["reproduction_max_abs_diff"].values()),
                       "ridge_s": d["ridge_varscale_s"], "lstm_s": d["lstm_varscale_s"],
                       **{f"{m}_cal_cov90": d[f"{m}_cal_native_cov90_engine_weighted"] for m in MODELS}} for d in meta])
    M.to_csv(TAB / "recal_runs.csv", index=False)
    return V, P, Bd, Tt, M


def lstm_tail_diagnostic():
    """Post-plan diagnostic (not in the D7 plan): why variance scaling over-widens the LSTM intervals.
    Compares the maximum-likelihood scale factor with the factor that would give 90% coverage on the
    calibration windows, and the share of windows with |z| > 3 (Gaussian: 0.27%)."""
    import sys
    from scipy.stats import norm
    import run_main as RM
    sys.path.insert(0, str(ROOT.parents[0] / "src"))
    from data import load, regime_stats, normalise, windows
    import conformal as C
    rows = []
    for fd in SUBSETS:
        S = load(fd, "train")
        for f in sorted(RAWR.glob(f"{fd}_seed*.npz")):
            seed = int(f.stem.split("seed")[1]); r = np.load(f)
            fit_u, val_u, cal_u = RM.split_engines(S.unit.values, seed)
            Xs, _ = normalise(S, regime_stats(S[S.unit.isin(np.r_[fit_u, val_u])]))
            m = S.unit.isin(cal_u).values
            _, _, _, uc = windows(S[m].reset_index(drop=True), Xs[m])
            w = C.unit_weights(uc); w = w / w.sum()
            z = norm.ppf(np.clip(r["lstm_cal_pit"], 1e-12, 1 - 1e-12))
            o = np.argsort(np.abs(z))
            q90 = np.abs(z)[o][np.searchsorted(np.cumsum(w[o]), 0.9)]
            meta = json.loads(f.with_suffix(".json").read_text())
            rows.append({"subset": fd, "seed": seed, "s_mle": meta["lstm_varscale_s"],
                         "s_needed_for_90pct": float(q90 / norm.ppf(0.95)), "share_abs_z_gt_3": float(np.sum(w * (np.abs(z) > 3)))})
    D = pd.DataFrame(rows); D.to_csv(TAB / "recal_lstm_tail_diagnostic.csv", index=False)
    return D


def figure(V):
    plt = AM.plt
    style = {"native": dict(ls=":", marker="o"), "conformal": dict(ls="--", marker="s"), "iso": dict(ls="-", marker="D")}
    fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.8), sharex=True, sharey=True, constrained_layout=True)
    for i, fd in enumerate(SUBSETS):
        for j, m in enumerate(MODELS):
            ax = axes[i, j]; ax.plot([0.45, 1], [0.45, 1], color=AM.MUTED, lw=1)
            for meth, st in style.items():
                g = V[(V.subset == fd) & (V.model == m) & (V.method == meth) & (V.metric == "PICP")].sort_values("level")
                if meth == "iso":
                    ax.fill_between(g.level, g.lo95, g.hi95, color=AM.COL[m], alpha=0.15, lw=0)
                ax.plot(g.level, g.value, color=AM.COL[m] if meth == "iso" else AM.INK2, lw=2 if meth == "iso" else 1.3,
                        ms=4.5, label=METHODS[meth], **st)
            ax.set_title(f"{fd} — {AM.NAME[m]}"); ax.set_xlim(0.45, 1.0); ax.set_ylim(0.2, 1.0)
            if i == 1: ax.set_xlabel("nominal coverage")
            if j == 0: ax.set_ylabel("empirical coverage (test engines)")
    from matplotlib.lines import Line2D
    hs = [Line2D([], [], color=AM.INK2, lw=1.3, **style["native"], ms=5, label="native"),
          Line2D([], [], color=AM.INK2, lw=1.3, **style["conformal"], ms=5, label="split conformal"),
          Line2D([], [], color=AM.INK2, lw=2, **style["iso"], ms=5, label="isotonic recalibration (band: 95% engine bootstrap)")]
    fig.legend(handles=hs, loc="outside lower center", ncol=3)
    fig.suptitle("Exploratory (D7): calibration curves after post-hoc isotonic recalibration; solid grey line = perfect calibration",
                 color=AM.INK)
    AM.save(fig, "fig5_recalibration")


if __name__ == "__main__":
    V, P, Bd, Tt, M = run()
    figure(V)
    pd.set_option("display.width", 220)
    print(lstm_tail_diagnostic().round(3).to_string(index=False))
    print(M.round(3).to_string(index=False))
    print(V[V.metric == "CE"].round(3).to_string(index=False))
    print(V[(V.metric != "CE") & (V.level == 0.9)].round(3).to_string(index=False))
    print(P.round(3).to_string(index=False))
    print(Bd.round(3).to_string(index=False))
    print(Tt[Tt.level.isin([0.8, 0.9, 0.95])].round(3).to_string(index=False))

In [ ]:
%%writefile /content/RUL_project/main/run_uq.py
"""Exploratory deviation D8 (see DEVIATIONS.md): epistemic uncertainty and locally adaptive conformal.

For one (subset, seed), with the main study's splits:
  * deep ensemble of 5 beta-NLL LSTMs (member 0 = the main-study LSTM);
  * MC dropout (T = 50) on the main-study LSTM;
  * CQR on the gradient-boosting native quantiles;
  * normalised conformal (|y - mu| / sigma) for the single LSTM, the ensemble and MC dropout;
  * standard split conformal for the ensemble and MC-dropout means.
Saves results/raw_uq/{FD}_seed{s}.npz and .json. No metric is computed here.

Usage: python run_uq.py FD001 0
"""
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from scipy.stats import norm

import run_main as RM
from run_main import LEVELS, QUANTS, SCALE

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from data import load, regime_stats, normalise, windows  # noqa: E402
from models import handcrafted  # noqa: E402
import conformal as C  # noqa: E402

ROOT = Path(__file__).resolve().parent
OUT = ROOT / "results" / "raw_uq"
M_ENS, T_MC = 5, 50
QI = {q: i for i, q in enumerate(QUANTS)}


def cq(scores, w, L):
    """Engine-weighted conformal quantile of arbitrary non-conformity scores (same correction as the main study)."""
    return float(C._wq_aug(scores[None, :], w, np.ones(1), L)[0])


@torch.no_grad()
def predict_mc(m, X, T, seed, bs=4096):
    """MC dropout: dropout layers in train mode, everything else as in eval. Returns (T, n) means and variances."""
    torch.manual_seed(seed)
    m.eval()
    m.lstm.train()  # nn.LSTM applies its inter-layer dropout only in training mode
    mus, vs = [], []
    for _ in range(T):
        a, b = [], []
        for i in range(0, len(X), bs):
            mu, lv = m(torch.from_numpy(X[i:i + bs]))
            a.append(mu.numpy() * SCALE); b.append(lv.exp().numpy() * SCALE ** 2)
        mus.append(np.concatenate(a)); vs.append(np.concatenate(b))
    m.eval()
    return np.stack(mus), np.stack(vs)


def mixture(mus, vs):
    mu = mus.mean(0)
    alea, epi = vs.mean(0), mus.var(0)
    return mu, alea, epi


def main(fd, seed):
    t0 = time.time()
    OUT.mkdir(parents=True, exist_ok=True)
    saved = dict(np.load(ROOT / "results" / "raw" / f"{fd}_seed{seed}.npz"))
    S, T = load(fd, "train"), load(fd, "test")
    fit_u, val_u, cal_u = RM.split_engines(S.unit.values, seed)
    stats = regime_stats(S[S.unit.isin(np.r_[fit_u, val_u])])
    Xs, _ = normalise(S, stats)
    Xt, _ = normalise(T, stats)

    def sub(units):
        m = S.unit.isin(units).values
        return windows(S[m].reset_index(drop=True), Xs[m])

    (Xf, yf, _, _), (Xv, yv, _, _), (Xc, yc, _, uc) = sub(fit_u), sub(val_u), sub(cal_u)
    Xte, yte, _, ute = windows(T, Xt, last_only=True)
    Ff, Fv, Fc, Fte = map(handcrafted, (Xf, Xv, Xc, Xte))
    w = C.unit_weights(uc)
    out = {"y": yte, "unit": ute}
    meta = {"subset": fd, "seed": seed, "M": M_ENS, "T": T_MC}

    # ---------------- gradient boosting: CQR on the native quantiles
    gp, gq, _ = RM.fit_lgbm(Ff, yf, Fv, yv, seed)
    check = {"gbm": float(np.max(np.abs(gp(Fte) - saved["gbm_point"])))}
    Qc, Qt = gq(Fc), gq(Fte)
    for L in LEVELS:
        jl, jh = QI[round((1 - L) / 2, 3)], QI[round((1 + L) / 2, 3)]
        E = np.maximum(Qc[:, jl] - yc, yc - Qc[:, jh])
        q = cq(E, w, L)
        out[f"gbm_adapt_lo_{L}"], out[f"gbm_adapt_hi_{L}"] = Qt[:, jl] - q, Qt[:, jh] + q
        out[f"gbm_adapt_q_{L}"] = np.array(q)

    # ---------------- LSTM ensemble (member 0 = main-study network)
    members = []
    for k in range(M_ENS):
        net, info = RM.train_lstm(Xf, yf, Xv, yv, seed=1000 + seed + 100 * k)
        members.append(net)
        meta[f"member{k}_epochs"] = info["epochs"]
    mu0 = RM.predict_lstm(members[0], Xte)[0]
    check["lstm"] = float(np.max(np.abs(mu0 - saved["lstm_point"])))
    meta["reproduction_max_abs_diff"] = check
    if max(check.values()) > 1e-3:
        raise RuntimeError(f"re-fit does not reproduce the main study: {check}")

    def preds(X):
        P = [RM.predict_lstm(n, X) for n in members]
        return np.stack([p[0] for p in P]), np.stack([p[1] for p in P])

    ens_c, ens_t = mixture(*preds(Xc)), mixture(*preds(Xte))
    mc_c = mixture(*predict_mc(members[0], Xc, T_MC, seed=7000 + seed))
    mc_t = mixture(*predict_mc(members[0], Xte, T_MC, seed=8000 + seed))
    single_c, single_t = RM.predict_lstm(members[0], Xc), RM.predict_lstm(members[0], Xte)
    single_c = (single_c[0], single_c[1], np.zeros_like(single_c[1]))
    single_t = (single_t[0], single_t[1], np.zeros_like(single_t[1]))

    for name, (mc_, ac, ec), (mt, at, et) in (("ens", ens_c, ens_t), ("mcd", mc_c, mc_t), ("lstm", single_c, single_t)):
        sc, st = np.sqrt(ac + ec), np.sqrt(at + et)
        if name != "lstm":
            out[f"{name}_point"] = mt
            out[f"{name}_alea_var"], out[f"{name}_epi_var"] = at, et
            out[f"{name}_cal_point"] = mc_
        for L in LEVELS:
            z = norm.ppf(.5 + L / 2)
            if name != "lstm":
                out[f"{name}_native_lo_{L}"], out[f"{name}_native_hi_{L}"] = mt - z * st, mt + z * st
                out[f"{name}_cq_{L}"] = np.array(cq(np.abs(yc - mc_), w, L))
            qn = cq(np.abs(yc - mc_) / sc, w, L)
            out[f"{name}_adapt_lo_{L}"], out[f"{name}_adapt_hi_{L}"] = mt - qn * st, mt + qn * st
            out[f"{name}_adapt_q_{L}"] = np.array(qn)
        meta[f"{name}_test_mean_sd"] = float(np.mean(st))
        meta[f"{name}_test_epistemic_share"] = float(np.mean(et / (at + et)))

    np.savez_compressed(OUT / f"{fd}_seed{seed}.npz", **out)
    meta["seconds"] = time.time() - t0
    (OUT / f"{fd}_seed{seed}.json").write_text(json.dumps(meta, indent=1, default=float))
    print(f"[{fd} s{seed}] uq done {meta['seconds']:.0f}s repro={check} "
          f"epi_share ens={meta['ens_test_epistemic_share']:.3f} mcd={meta['mcd_test_epistemic_share']:.3f}", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], int(sys.argv[2]))

In [ ]:
%%writefile /content/RUL_project/main/analyze_uq.py
"""Exploratory deviation D8: evaluation of deep ensembles, MC dropout and locally adaptive conformal.
Same evaluation as the main study (clipping to [0,125], seed-averaged per-engine values, engine bootstrap).
Writes results/tables/uq_*.csv and results/figures/fig6_uq.*   Usage: python analyze_uq.py"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

import analyze_main as AM
from inference import load_arrays, diff_ci, value_ci, resolved, LEVELS, HIGH, SUBSETS, CAP, BANDS

ROOT = Path(__file__).resolve().parent
RAWU, TAB, FIG = ROOT / "results" / "raw_uq", ROOT / "results" / "tables", ROOT / "results" / "figures"
ALL = ["ridge", "gbm", "lstm", "ens", "mcd"]
NAME = {**AM.NAME, "ens": "LSTM deep ensemble (M=5)", "mcd": "LSTM MC dropout (T=50)"}
COL = {**AM.COL, "ens": "#7c5cd6", "mcd": "#b8860b"}
METHS = {"ridge": ["native", "conformal"], "gbm": ["native", "conformal", "adapt"],
         "lstm": ["native", "conformal", "adapt"], "ens": ["native", "conformal", "adapt"], "mcd": ["native", "conformal", "adapt"]}
ADAPT = {"gbm": "CQR", "lstm": "normalised conformal", "ens": "normalised conformal", "mcd": "normalised conformal"}


def load_all(fd):
    A, y = load_arrays(fd)
    main = sorted((ROOT / "results" / "raw").glob(f"{fd}_seed*.npz"))
    uq = sorted(RAWU.glob(f"{fd}_seed*.npz"))
    assert [f.name for f in main] == [f.name for f in uq], "D8 runs missing"
    for fm, fu in zip(main, uq):
        z, r = dict(np.load(fm)), dict(np.load(fu))
        assert np.array_equal(z["unit"], r["unit"]) and np.array_equal(z["y"], y)
        for m in ("ens", "mcd"):
            p = r[f"{m}_point"]; pc = np.clip(p, 0, CAP)
            A.setdefault((m, "abs"), []).append(np.abs(pc - y))
            A.setdefault((m, "sq"), []).append((pc - y) ** 2)
            A.setdefault((m, "abs_raw"), []).append(np.abs(p - y))
        for m in ALL:
            for meth in METHS[m]:
                for L in LEVELS:
                    if m in ("ens", "mcd") and meth == "native":
                        lo, hi = r[f"{m}_native_lo_{L}"], r[f"{m}_native_hi_{L}"]
                    elif m in ("ens", "mcd") and meth == "conformal":
                        q = float(r[f"{m}_cq_{L}"]); lo, hi = r[f"{m}_point"] - q, r[f"{m}_point"] + q
                    elif meth == "adapt":
                        lo, hi = r[f"{m}_adapt_lo_{L}"], r[f"{m}_adapt_hi_{L}"]
                    elif meth == "native":
                        lo, hi = z[f"{m}_native_lo_{L}"], z[f"{m}_native_hi_{L}"]
                    else:
                        q = float(z[f"{m}_cq_{L}"]); lo, hi = z[f"{m}_point"] - q, z[f"{m}_point"] + q
                    lc, hc = np.clip(lo, 0, CAP), np.clip(hi, 0, CAP)
                    if m in ("ens", "mcd") or meth == "adapt":
                        A.setdefault((m, "cov", meth, L), []).append(((y >= lc) & (y <= hc)).astype(float))
                        A.setdefault((m, "w", meth, L), []).append(hc - lc)
                    A.setdefault((m, "is", meth, L), []).append(AM.interval_score(lc, hc, y, 1 - L))
    A = {k: (np.stack(v) if isinstance(v, list) else v) for k, v in A.items()}
    return A, y


def boot_mean(X, seed=0):
    rng = np.random.default_rng(seed)
    b = X[rng.integers(0, len(X), (2000, len(X)))].mean(1)
    return float(X.mean()), float(np.quantile(b, .025)), float(np.quantile(b, .975))


def method_diff_ci(A, n_s, n_e, m, ma, mb, what, L=None, mask=None):
    B = dict(A)
    for l in LEVELS:
        for kind in ("cov", "w"):
            B[("__a", kind, "x", l)] = A[(m, kind, ma, l)]
            B[("__b", kind, "x", l)] = A[(m, kind, mb, l)]
    return diff_ci(B, n_s, n_e, "__a", "__b", what, "x", L, mask=mask)


def row(fd, what, d, **kw):
    return {"subset": fd, **kw, "metric": what, "diff": d[0], "lo95": d[1], "hi95": d[2], "resolved": resolved(*d[1:])}


def run():
    vals, pairs, bands = [], [], []
    for fd in SUBSETS:
        A, y = load_all(fd)
        n_s, n_e = A[("ridge", "abs")].shape
        for m in ALL:
            for what in ("RMSE", "MAE"):
                v = value_ci(A, n_s, n_e, m, what)
                vals.append({"subset": fd, "model": m, "method": "point", "metric": what, "level": np.nan,
                             "value": v[0], "lo95": v[1], "hi95": v[2]})
            for meth in METHS[m]:
                v = value_ci(A, n_s, n_e, m, "CE", meth)
                vals.append({"subset": fd, "model": m, "method": meth, "metric": "CE", "level": np.nan,
                             "value": v[0], "lo95": v[1], "hi95": v[2]})
                for L in LEVELS:
                    for what in ("PICP", "MPIW"):
                        v = value_ci(A, n_s, n_e, m, what, meth, L)
                        vals.append({"subset": fd, "model": m, "method": meth, "metric": what, "level": L,
                                     "value": v[0], "lo95": v[1], "hi95": v[2]})
                    v = boot_mean(A[(m, "is", meth, L)].mean(0))
                    vals.append({"subset": fd, "model": m, "method": meth, "metric": "IS", "level": L,
                                 "value": v[0], "lo95": v[1], "hi95": v[2]})
                for lo_b, hi_b in BANDS:
                    mask = (y > lo_b) & (y <= hi_b) if lo_b > 0 else (y <= hi_b)
                    bands.append({"subset": fd, "model": m, "method": meth, "band": f"{lo_b}-{hi_b}", "n": int(mask.sum()),
                                  "PICP90": float(A[(m, "cov", meth, 0.90)][:, mask].mean()),
                                  "MPIW90": float(A[(m, "w", meth, 0.90)][:, mask].mean())})
        # (i) native CE and (ii) RMSE between models
        for a, b in (("ens", "lstm"), ("mcd", "lstm"), ("ens", "gbm")):
            pairs.append(row(fd, "CE", diff_ci(A, n_s, n_e, a, b, "CE", "native"), A=a, B=b, method="native", level=np.nan))
        for a, b in (("ens", "lstm"), ("ens", "gbm"), ("mcd", "lstm")):
            pairs.append(row(fd, "RMSE", diff_ci(A, n_s, n_e, a, b, "RMSE"), A=a, B=b, method="point", level=np.nan))
        # (iii) adaptive vs standard conformal per model
        for m in ("gbm", "lstm", "ens", "mcd"):
            pairs.append(row(fd, "CE", method_diff_ci(A, n_s, n_e, m, "adapt", "conformal", "CE"),
                             A=f"{m}:adapt", B=f"{m}:conformal", method="-", level=np.nan))
            for L in HIGH:
                pairs.append(row(fd, "MPIW", method_diff_ci(A, n_s, n_e, m, "adapt", "conformal", "MPIW", L),
                                 A=f"{m}:adapt", B=f"{m}:conformal", method="-", level=L))
                d = A[(m, "is", "adapt", L)].mean(0) - A[(m, "is", "conformal", L)].mean(0)
                pairs.append(row(fd, "IS", boot_mean(d), A=f"{m}:adapt", B=f"{m}:conformal", method="-", level=L))
            pairs.append(row(fd, "PICP_RUL<=40", method_diff_ci(A, n_s, n_e, m, "adapt", "conformal", "PICP", 0.90, mask=y <= 40),
                             A=f"{m}:adapt", B=f"{m}:conformal", method="-", level=0.90))
    V, P, Bd = pd.DataFrame(vals), pd.DataFrame(pairs), pd.DataFrame(bands)
    V.to_csv(TAB / "uq_values.csv", index=False); P.to_csv(TAB / "uq_paired.csv", index=False)
    Bd.to_csv(TAB / "uq_bands.csv", index=False)
    meta = [json.loads(f.read_text()) for f in sorted(RAWU.glob("*.json"))]
    M = pd.DataFrame([{"subset": d["subset"], "seed": d["seed"], "repro_max": max(d["reproduction_max_abs_diff"].values()),
                       "ens_epistemic_share": d["ens_test_epistemic_share"], "mcd_epistemic_share": d["mcd_test_epistemic_share"],
                       "lstm_mean_sd": d["lstm_test_mean_sd"], "ens_mean_sd": d["ens_test_mean_sd"], "mcd_mean_sd": d["mcd_test_mean_sd"],
                       "member_epochs": "/".join(str(d[f"member{k}_epochs"]) for k in range(d["M"])), "seconds": d["seconds"]}
                      for d in meta])
    M.to_csv(TAB / "uq_runs.csv", index=False)
    return V, P, Bd, M


def figure(V, Bd):
    plt = AM.plt
    fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.0), constrained_layout=True)
    show = ["gbm", "lstm", "ens", "mcd"]
    for i, fd in enumerate(SUBSETS):
        ax = axes[i, 0]; ax.plot([0.45, 1], [0.45, 1], color=AM.MUTED, lw=1)
        for m in show:
            g = V[(V.subset == fd) & (V.model == m) & (V.method == "native") & (V.metric == "PICP")].sort_values("level")
            ax.fill_between(g.level, g.lo95, g.hi95, color=COL[m], alpha=0.12, lw=0)
            ax.plot(g.level, g.value, color=COL[m], lw=2, marker="o", ms=4.5, label=NAME[m])
        ax.set_xlim(0.45, 1.0); ax.set_ylim(0.2, 1.0); ax.set_title(f"{fd}: native intervals")
        ax.set_ylabel("empirical coverage (test engines)")
        if i == 1: ax.set_xlabel("nominal coverage")
        ax = axes[i, 1]
        x = np.arange(len(show))
        for k, (meth, alpha) in enumerate((("conformal", 0.4), ("adapt", 1.0))):
            g = Bd[(Bd.subset == fd) & (Bd.method == meth) & (Bd.band == "0-40")].set_index("model")
            wv, cv = g.loc[show, "MPIW90"].values, g.loc[show, "PICP90"].values
            ax.bar(x + (k - 0.5) * 0.38, wv, 0.36, color=[COL[m] for m in show], alpha=alpha, edgecolor=AM.SURF, lw=2)
            for xi, wi, ci in zip(x, wv, cv):
                ax.text(xi + (k - 0.5) * 0.38, wi + 1.2, f"{ci:.2f}", ha="center", va="bottom", fontsize=7.5, color=AM.INK2)
        ax.set_xticks(x, ["GBM", "LSTM", "Ensemble", "MC dropout"]); ax.set_ylim(0, 105)
        ax.set_ylabel("mean 90% interval width (cycles)")
        ax.set_title(f"{fd}: near failure (true RUL \u2264 40), n = {int(g.n.iloc[0])} engines")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="outside lower center", ncol=4)
    fig.suptitle("Exploratory (D8). Left: native calibration curves (band = 95% engine bootstrap). Right: width near failure, "
                 "light = split conformal, solid = adaptive (CQR for GBM, normalised for LSTMs); labels = coverage", color=AM.INK, fontsize=9)
    AM.save(fig, "fig6_uq")


if __name__ == "__main__":
    V, P, Bd, M = run()
    figure(V, Bd)
    pd.set_option("display.width", 220); pd.set_option("display.max_rows", 500)
    print(M.round(3).to_string(index=False))
    print(V[V.metric.isin(["RMSE", "MAE", "CE"])].round(3).to_string(index=False))
    print(V[(V.metric.isin(["PICP", "MPIW", "IS"])) & (V.level == 0.9)].round(3).to_string(index=False))
    print(P.round(3).to_string(index=False))
    print(Bd.round(3).to_string(index=False))

## 3. Data: download and verify

In [ ]:
import hashlib, shutil
from huggingface_hub import hf_hub_download

EXPECTED_SHA256 = {
 "train_FD001.txt": "963b5e22825b34d8b21c69e1aeb4af3e647050eb672ee8834ba4b5d91d2de0f8",
 "test_FD001.txt": "3cda7109ce17bafb5443f2ac926cfcf88154b941b8c4cf95eb55d1ddd6f52851",
 "RUL_FD001.txt": "a19c8ec94931949d0485bdc35118206e9c81c4547b422efb9cf86f4ceddbceca",
 "train_FD004.txt": "27ef6160b6a1dcb2613a88de9c239f763b223f02cdc41dc5cdedc5dc189b6218",
 "test_FD004.txt": "1dc675fff0624bac10786927c6715b37d1297657137400d2b1a3138d777a3ba5",
 "RUL_FD004.txt": "196b836b85a95ac7fdbbf29c5fdf1657382eafa445644d114ffaaf50dc2975e1"
}
RAW = "/content/RUL_project/data_raw"
for f in EXPECTED_SHA256:
    p = hf_hub_download("SoyVitou/NASA-C-MAPSS-Turbofan-Engine", f"data/{f}", repo_type="dataset")
    shutil.copy(p, f"{RAW}/{f}")
    h = hashlib.sha256(open(f"{RAW}/{f}", "rb").read()).hexdigest()
    print(f"{f:16s}", "OK" if h == EXPECTED_SHA256[f] else "CHECKSUM MISMATCH")

In [ ]:
# Check against the official C-MAPSS specification and the official RUL files
sys.path.insert(0, "/content/RUL_project/src")
from data import load, regime_stats, normalise, windows
expected = {"FD001": {"train": (20631, 100), "test": (13096, 100)}, "FD004": {"train": (61249, 249), "test": (41214, 248)}}
for fd, parts in expected.items():
    for split, (rows, units) in parts.items():
        d = load(fd, split)
        assert (len(d), d.unit.nunique()) == (rows, units), (fd, split, len(d), d.unit.nunique())
    t = load(fd, "test"); X, _ = normalise(t, regime_stats(load(fd, "train")))
    _, _, y_raw, _ = windows(t, X, last_only=True)
    official = pd.read_csv(f"/content/RUL_project/data_raw/RUL_{fd}.txt", header=None)[0].values
    assert np.allclose(y_raw, official)
    print(fd, "rows, engines and last-window RUL match the official files")

In [ ]:
# Integrity of the pre-registration (should match the checksum recorded on 16 Nov 2026)
h = hashlib.sha256(open("/content/RUL_project/main/PREREGISTRATION.md", "rb").read()).hexdigest()
print(h, "MATCHES" if h == "c85a61f76da376f290b381de926bd42844d11a8a620c4894c7f09d08b4b69ee5" else "DIFFERS")

## 4. Sanity check of the conformal code
On synthetic grouped data the engine-weighted split-conformal interval should cover at least the nominal level.

In [ ]:
%cd /content/RUL_project/main
sys.path.insert(0, "/content/RUL_project/main")
from run_main import conformal_q
rng = np.random.default_rng(0)
for L in (0.8, 0.9, 0.95):
    cov = []
    for _ in range(400):
        n = 30; eff = rng.normal(0, 3, n); u = np.repeat(np.arange(n), rng.integers(5, 60, n))
        y = eff[u] + rng.normal(0, 1, len(u)); q = conformal_q(np.zeros_like(y), y, u, L)
        cov.append(abs(rng.normal(0, 3) + rng.normal(0, 1)) <= q)
    print(f"nominal {L:.2f} -> empirical {np.mean(cov):.3f}")
print("5 engines at 90% ->", conformal_q(np.zeros(50), rng.normal(size=50), np.repeat(np.arange(5), 10), 0.9), "(must be inf)")

## 5. Run the experiments (2 subsets × 5 seeds)
Each run trains the three models and saves test-set predictions to `results/raw/`.

In [ ]:
%cd /content/RUL_project/main
!for fd in FD001 FD004; do for s in 0 1 2 3 4; do python run_main.py $fd $s 2>&1 | grep -E "done|Error|Traceback"; done; done

## 6. Analysis, hypothesis tests and supporting checks

In [ ]:
%cd /content/RUL_project/main
!python analyze_main.py 2>&1 | tail -1
!python inference.py
!python extra_checks.py 2>&1 | tail -5

## 6b. Exploratory: post-hoc recalibration (deviation D7)
Re-fits the models (checking that the predictions reproduce the main study) and builds isotonic-recalibrated and variance-scaled intervals on the calibration engines. About 10–15 minutes.

In [ ]:
%cd /content/RUL_project/main
!for fd in FD001 FD004; do for s in 0 1 2 3 4; do python run_recal.py $fd $s 2>&1 | grep -E "done|Error|Traceback"; done; done
!python analyze_recal.py > /dev/null && echo "recalibration analysis done"

## 6c. Exploratory: deep ensemble, MC dropout and adaptive conformal (deviation D8)
Trains four extra LSTMs per run (ensemble of 5), runs MC dropout on the main LSTM, and builds CQR / normalised conformal intervals. About 25–40 minutes on the free CPU runtime.

In [ ]:
%cd /content/RUL_project/main
!for fd in FD001 FD004; do for s in 0 1 2 3 4; do python run_uq.py $fd $s 2>&1 | grep -E "done|Error|Traceback"; done; done
!python analyze_uq.py > /dev/null && echo "D8 analysis done"

## 7. Key results

In [ ]:
TAB = "/content/RUL_project/main/results/tables"
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 200)
D = pd.read_csv(f"{TAB}/metrics_per_seed.csv")
print("Point accuracy (mean over 5 seeds):")
display(D[D.method == "point"].groupby(["subset", "model"])[["MAE", "RMSE"]].mean().round(2))
print("Coverage (PICP) and width (MPIW) at 80/90/95%:")
I = D[(D.method != "point") & D.level.isin([0.8, 0.9, 0.95])]
display(I.groupby(["subset", "model", "method", "level"])[["PICP", "MPIW", "IS"]].mean().round(3).unstack("level"))
print("Exploratory (D7): calibration error of native, conformal, isotonic-recalibrated and variance-scaled intervals:")
display(pd.read_csv(f"{TAB}/recal_values.csv").query("metric == 'CE'").pivot_table(index=["subset", "model"], columns="method", values="value").round(3))
print("Exploratory (D8): accuracy and calibration error incl. deep ensemble (ens), MC dropout (mcd) and adaptive conformal (adapt):")
U = pd.read_csv(f"{TAB}/uq_values.csv")
display(U[U.metric == "RMSE"].pivot_table(index="subset", columns="model", values="value").round(2))
display(U.query("metric == 'CE'").pivot_table(index=["subset", "model"], columns="method", values="value").round(3))
print("Hypothesis tests (paired engine bootstrap):")
display(pd.read_csv(f"{TAB}/hypotheses_primary.csv")[["hypothesis", "subset", "level", "result"]])

In [ ]:
from IPython.display import Image, display
FIG = "/content/RUL_project/main/results/figures"
for f in ("fig1_calibration_curves", "fig2_width", "fig3_accuracy_vs_calibration", "fig4_bands", "fig5_recalibration", "fig6_uq"):
    display(Image(f"{FIG}/{f}.png", width=900))

## 8. Download all results

In [ ]:
!cd /content && zip -qr RUL_project_results.zip RUL_project/main RUL_project/src
from google.colab import files
files.download("/content/RUL_project_results.zip")